In [ ]:
# Multihazard Network Analysis v3 (CONUS 50 km Grid)
# March 26-April 8, 2023 event reconstruction scaffold.

print("Notebook scaffold initialized for March 26-April 8, 2023 event reconstruction.")
print("Workflow: AOI/network setup -> grid/hazard alignment -> projection to lines/substations -> diagnostics.")

In [ ]:
import os
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt

from shapely.geometry import Point, LineString, MultiLineString

# Event configuration
EVENT_NAME = "March26_April08_2023"
EVENT_START = pd.Timestamp("2023-03-26 00:00:00", tz="UTC")
EVENT_END = pd.Timestamp("2023-04-08 23:59:59", tz="UTC")

# Core paths (update as needed)
PATH_POWERGRID = Path("/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/physical_grid_data/U.S._Electric_Power_Transmission_Lines.geojson")
PATH_AOI = Path("/Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience/data/selected_states_MarchApril2023_event.geojson")
PATH_ANALYSIS_GRID = Path("/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/analysis_grid_CONUS_50km.nc")

# Hazard data sources
DIR_TERRESTRIAL_WEATHER = Path("/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/terrestrial_weather")
DIR_WILDFIRE = Path("/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/wildfire")
PATH_SPACE_WEATHER = Path("/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/Lucas_GeoElectricFields_March-April2023/mcgranaghan-rawdata.nc")

# Hazard paths/config used by hazard ingest functions
HAZARD_PATHS = {
    "terrestrial_weather_dir": DIR_TERRESTRIAL_WEATHER,
    "wildfire_dir": DIR_WILDFIRE,
    "space_weather_file": PATH_SPACE_WEATHER,
}

# Outage/disruption data placeholders
OBS_PATHS = {
    "eagle_i": None,
    "myriad": None,
}

print(f"Event: {EVENT_NAME}")
print(f"Time window: {EVENT_START} to {EVENT_END}")
print(f"AOI path exists: {PATH_AOI.exists()}")
print(f"Power grid path exists: {PATH_POWERGRID.exists()}")
print(f"Analysis grid path exists: {PATH_ANALYSIS_GRID.exists()}")
print(f"Terrestrial weather dir exists: {DIR_TERRESTRIAL_WEATHER.exists()}")
print(f"Wildfire dir exists: {DIR_WILDFIRE.exists()}")
print(f"Wildfire daily dir exists: {(DIR_WILDFIRE / 'daily').exists()}")
print(f"Space weather file exists: {PATH_SPACE_WEATHER.exists()}")

## Step 1: Load AOI and Transmission Network Base

Load AOI and transmission lines, harmonize CRS, and clip network geometry to the event AOI.

In [ ]:
gdf_powergrid = gpd.read_file(PATH_POWERGRID)
gdf_aoi = gpd.read_file(PATH_AOI)

# Harmonize CRS
gdf_aoi = gdf_aoi.to_crs(gdf_powergrid.crs)

# Keep only rows with usable endpoint substation names
for col in ["SUB_1", "SUB_2"]:
    if col in gdf_powergrid.columns:
        gdf_powergrid = gdf_powergrid[~gdf_powergrid[col].isin(["NOT AVAILABLE", "NONE"])]

# Clip network to AOI
gdf_powergrid_aoi = gpd.sjoin(gdf_powergrid, gdf_aoi, how="inner", predicate="intersects")
if "index_right" in gdf_powergrid_aoi.columns:
    gdf_powergrid_aoi = gdf_powergrid_aoi.drop(columns=["index_right"])

print(f"AOI features: {len(gdf_aoi):,}")
print(f"Power lines (full): {len(gdf_powergrid):,}")
print(f"Power lines in AOI: {len(gdf_powergrid_aoi):,}")
print(f"Power grid CRS: {gdf_powergrid.crs}")

## Step 2: Load Common Analysis Grid

Load the CONUS analysis grid that serves as the shared spatial index for all hazards and downstream stress/disruption mapping.

In [ ]:
from scipy.spatial import cKDTree


def _to_naive_timestamp(ts):
    ts = pd.Timestamp(ts)
    return ts.tz_convert(None) if ts.tzinfo is not None else ts


def _daily_dates(event_start, event_end):
    start = _to_naive_timestamp(event_start).normalize()
    end = _to_naive_timestamp(event_end).normalize()
    return pd.date_range(start=start, end=end, freq="D")


def _resolve_daily_dir(base_dir):
    base_dir = Path(base_dir)
    daily_dir = base_dir / "daily"
    return daily_dir if daily_dir.exists() else base_dir


def _build_daily_file_list(base_dir, prefix, dates):
    base_dir = _resolve_daily_dir(base_dir)
    files = [base_dir / f"{prefix}_{d.strftime('%Y%m%d')}.nc" for d in dates]
    existing = [p for p in files if p.exists()]
    missing = [p for p in files if not p.exists()]
    return existing, missing


def _open_daily_netcdfs(file_list):
    if not file_list:
        return None
    datasets = [xr.open_dataset(fp) for fp in file_list]
    if len(datasets) == 1:
        return datasets[0]
    return xr.concat(datasets, dim="time", data_vars="minimal", coords="minimal", compat="override")


def _subset_to_event_time(ds, event_start, event_end):
    if ds is None or "time" not in ds.coords:
        return ds
    start = _to_naive_timestamp(event_start)
    end = _to_naive_timestamp(event_end)
    return ds.sel(time=slice(start, end))


def _coerce_to_target_grid(ds, ds_target_grid):
    """Ensure an already-gridded dataset is on target x/y coordinates."""
    if ds is None:
        return None
    if not {"x", "y"}.issubset(set(ds.dims)):
        return ds

    if ds.sizes.get("x") == ds_target_grid.sizes.get("x") and ds.sizes.get("y") == ds_target_grid.sizes.get("y"):
        out = ds.assign_coords(x=ds_target_grid["x"], y=ds_target_grid["y"])
    else:
        out = ds.interp(x=ds_target_grid["x"], y=ds_target_grid["y"], method="nearest")

    out = out.assign_coords(
        lat=(("y", "x"), ds_target_grid["lat"].values),
        lon=(("y", "x"), ds_target_grid["lon"].values),
    )
    return out


def _grid_space_weather_to_analysis_grid(ds_space_raw, ds_target_grid):
    """Map raw space-weather site values to nearest analysis-grid cells."""
    if ds_space_raw is None:
        return None

    ds_space = _subset_to_event_time(ds_space_raw, EVENT_START, EVENT_END)
    if "time" in ds_space.coords:
        ds_space = ds_space.resample(time="1h").mean()

    lat_name = "latitude" if "latitude" in ds_space.coords else "lat"
    lon_name = "longitude" if "longitude" in ds_space.coords else "lon"
    if lat_name not in ds_space.coords or lon_name not in ds_space.coords:
        raise ValueError("Space-weather dataset must include latitude/longitude coordinates.")

    site_dim = ds_space[lat_name].dims[0]
    target_lat = ds_target_grid["lat"].values.ravel()
    target_lon = ds_target_grid["lon"].values.ravel()
    ny, nx = ds_target_grid.sizes["y"], ds_target_grid.sizes["x"]
    ncells = ny * nx

    tree = cKDTree(np.column_stack([target_lat, target_lon]))
    src_points = np.column_stack([
        ds_space[lat_name].values.astype(float),
        ds_space[lon_name].values.astype(float),
    ])
    _, nearest_idx = tree.query(src_points, k=1)
    nearest_idx = nearest_idx.astype(np.int64)

    out_vars = {}
    for var_name, da in ds_space.data_vars.items():
        if "time" not in da.dims or site_dim not in da.dims:
            continue

        arr = da.transpose("time", site_dim).values.astype(float)
        nt = arr.shape[0]
        gridded = np.full((nt, ncells), np.nan, dtype=np.float32)

        for t in range(nt):
            row = arr[t]
            valid = np.isfinite(row)
            if not valid.any():
                continue
            sums = np.bincount(nearest_idx[valid], weights=row[valid], minlength=ncells)
            counts = np.bincount(nearest_idx[valid], minlength=ncells)
            mean_vals = np.full(ncells, np.nan, dtype=np.float32)
            nz = counts > 0
            mean_vals[nz] = (sums[nz] / counts[nz]).astype(np.float32)
            gridded[t, :] = mean_vals

        out_vars[var_name] = (("time", "y", "x"), gridded.reshape(nt, ny, nx))

    ds_out = xr.Dataset(
        data_vars=out_vars,
        coords={
            "time": ds_space["time"].values,
            "y": ds_target_grid["y"].values,
            "x": ds_target_grid["x"].values,
            "lat": (("y", "x"), ds_target_grid["lat"].values),
            "lon": (("y", "x"), ds_target_grid["lon"].values),
        },
        attrs={"gridding_method": "nearest_site_to_analysis_cell_hourly_mean"},
    )
    return ds_out


def read_hazard_layers(hazard_paths, event_start, event_end):
    """Read hazard datasets for event window.

    - terrestrial weather: daily gridded files named weather_CONUS_YYYYMMDD.nc
    - wildfire: daily gridded files named fires_analysis_grid_YYYYMMDD.nc
    - space weather: single non-gridded NetCDF file
    """
    dates = _daily_dates(event_start, event_end)

    weather_files, weather_missing = _build_daily_file_list(
        hazard_paths["terrestrial_weather_dir"], "weather_CONUS", dates
    )
    wildfire_files, wildfire_missing = _build_daily_file_list(
        hazard_paths["wildfire_dir"], "fires_analysis_grid", dates
    )

    ds_weather = _open_daily_netcdfs(weather_files)
    ds_wildfire = _open_daily_netcdfs(wildfire_files)

    # Ensure weather/wildfire are clipped to event time exactly.
    ds_weather = _subset_to_event_time(ds_weather, event_start, event_end)
    ds_wildfire = _subset_to_event_time(ds_wildfire, event_start, event_end)

    space_weather_path = Path(hazard_paths["space_weather_file"])
    ds_space_weather = xr.open_dataset(space_weather_path) if space_weather_path.exists() else None

    hazard_layers = {
        "terrestrial_weather": ds_weather,
        "wildfire": ds_wildfire,
        "space_weather_raw": ds_space_weather,
    }

    load_report = {
        "expected_days": len(dates),
        "weather_files_found": len(weather_files),
        "wildfire_files_found": len(wildfire_files),
        "weather_missing_count": len(weather_missing),
        "wildfire_missing_count": len(wildfire_missing),
        "space_weather_found": bool(ds_space_weather is not None),
    }

    return hazard_layers, load_report, weather_missing, wildfire_missing


def grid_hazard_layers(hazard_layers, ds_target_grid):
    """Grid all hazard datasets on the analysis grid."""
    if ds_target_grid is None:
        ds_target_grid = xr.open_dataset(PATH_ANALYSIS_GRID)

    weather_gridded = _coerce_to_target_grid(hazard_layers.get("terrestrial_weather"), ds_target_grid)
    wildfire_gridded = _coerce_to_target_grid(hazard_layers.get("wildfire"), ds_target_grid)
    space_weather_gridded = _grid_space_weather_to_analysis_grid(hazard_layers.get("space_weather_raw"), ds_target_grid)

    return {
        "terrestrial_weather_gridded": weather_gridded,
        "wildfire_gridded": wildfire_gridded,
        "space_weather_gridded": space_weather_gridded,
    }


if "ds_grid" not in globals() or ds_grid is None:
    ds_grid = xr.open_dataset(PATH_ANALYSIS_GRID)

hazard_layers, hazard_load_report, weather_missing_files, wildfire_missing_files = read_hazard_layers(
    HAZARD_PATHS, EVENT_START, EVENT_END
)
gridded_hazards = grid_hazard_layers(hazard_layers, ds_grid)

print("Hazard ingest and gridding complete.")
print(hazard_load_report)
for k, ds in gridded_hazards.items():
    if ds is None:
        print(f"{k}: None")
    else:
        print(f"{k}: dims={dict(ds.sizes)}")
if weather_missing_files:
    print(f"Missing terrestrial weather daily files: {len(weather_missing_files)}")
if wildfire_missing_files:
    print(f"Missing wildfire daily files: {len(wildfire_missing_files)}")

In [ ]:
# Compact schema diagnostics for loaded hazard datasets
for k, ds in hazard_layers.items():
    print(f"\n=== {k} ===")
    if ds is None:
        print("None")
        continue
    print("dims:", dict(ds.sizes))
    print("coords:", list(ds.coords)[:12])
    print("vars:", list(ds.data_vars)[:15])

## Hazard Snapshot Visualization (Timestamp)

Given a timestamp, render three figures (terrestrial weather, wildfire, space weather) on the analysis grid with AOI boundaries.

In [ ]:
# Choose snapshot timestamp (can be overridden before running this cell)
VIS_TIMESTAMP = pd.Timestamp("2023-03-30 12:00:00")


def _pick_var(ds, preferred=None):
    vars_list = list(ds.data_vars)
    if preferred and preferred in vars_list:
        return preferred
    for candidate in ["frp", "Eh", "Bh", "MAX_SHEAR"]:
        if candidate in vars_list:
            return candidate
    for v in vars_list:
        if v.lower() not in {"lat", "lon"}:
            return v
    return vars_list[0]


def _pick_var_with_data(ds, ts, preferred=None):
    if ds is None or "time" not in ds.coords:
        return None
    vars_list = list(ds.data_vars)
    ordered = []
    if preferred and preferred in vars_list:
        ordered.append(preferred)
    ordered.extend([v for v in vars_list if v not in ordered])

    for v in ordered:
        da = ds[v].sel(time=ts, method="nearest")
        arr = np.asarray(da.values, dtype=float)
        fill_value = da.attrs.get("_FillValue", None)
        if fill_value is not None:
            arr[np.isclose(arr, float(fill_value))] = np.nan
        good = arr[np.isfinite(arr)]
        if good.size and good.min() <= -900:
            arr[arr <= -900] = np.nan
        if np.isfinite(arr).any():
            return v
    return ordered[0] if ordered else None


def _robust_limits(vals):
    good = vals[np.isfinite(vals)]
    if good.size == 0:
        return None, None
    vmin = np.nanpercentile(good, 2)
    vmax = np.nanpercentile(good, 98)
    if np.isclose(vmin, vmax):
        vmin = np.nanmin(good)
        vmax = np.nanmax(good)
    return float(vmin), float(vmax)


def _plot_hazard_snapshot(ds, var_name, title, ts, gdf_aoi_local):
    if ds is None:
        print(f"{title}: dataset is None")
        return
    if "time" not in ds.coords:
        print(f"{title}: no time coordinate")
        return

    ts = pd.Timestamp(ts).tz_localize(None) if pd.Timestamp(ts).tzinfo else pd.Timestamp(ts)
    selected = ds[var_name].sel(time=ts, method="nearest")
    used_time = pd.Timestamp(selected["time"].values)

    if "lat" not in ds.coords or "lon" not in ds.coords:
        print(f"{title}: missing lat/lon coords")
        return

    lat2d = ds["lat"].values
    lon2d = ds["lon"].values
    vals = selected.values.astype(float)

    # Mask common nodata encodings before plotting
    fill_value = selected.attrs.get("_FillValue", None)
    if fill_value is not None:
        vals[np.isclose(vals, float(fill_value))] = np.nan
    vals[vals <= -900] = np.nan

    flat_lat = lat2d.ravel()
    flat_lon = lon2d.ravel()
    flat_vals = vals.ravel()
    valid = np.isfinite(flat_vals)

    fig, ax = plt.subplots(figsize=(10, 6))

    if valid.sum() == 0:
        if gdf_aoi_local is not None and not gdf_aoi_local.empty:
            gdf_aoi_local.boundary.plot(ax=ax, color="black", linewidth=1.0)
        ax.text(-95, 37.5, "No valid data at selected timestamp", ha="center", va="center", fontsize=12)
        ax.set_title(f"{title} | {var_name} | requested={ts} | used={used_time}")
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_xlim(-125, -65)
        ax.set_ylim(25, 50)
        plt.tight_layout()
        plt.show()
        print(f"{title}: all values are NaN at {used_time}")
        return

    vmin, vmax = _robust_limits(flat_vals)

    print(
        f"{title} stats | var={var_name} | used={used_time} | "
        f"n_valid={int(valid.sum())} | min={np.nanmin(flat_vals):.4g} | "
        f"max={np.nanmax(flat_vals):.4g} | mean={np.nanmean(flat_vals):.4g}"
    )

    sc = ax.scatter(
        flat_lon[valid],
        flat_lat[valid],
        c=flat_vals[valid],
        s=18,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        marker="s",
        linewidths=0,
        alpha=0.95,
    )
    plt.colorbar(sc, ax=ax, label=var_name)

    if gdf_aoi_local is not None and not gdf_aoi_local.empty:
        gdf_aoi_local.boundary.plot(ax=ax, color="black", linewidth=1.0)

    ax.set_title(f"{title} | {var_name} | requested={ts} | used={used_time}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_xlim(-125, -65)
    ax.set_ylim(25, 50)
    plt.tight_layout()
    plt.show()


# Ensure AOI is available in EPSG:4326 for overlay
if "gdf_aoi" in globals() and gdf_aoi is not None:
    gdf_aoi_plot = gdf_aoi.to_crs("EPSG:4326")
else:
    gdf_aoi_plot = gpd.read_file(PATH_AOI).to_crs("EPSG:4326")

# Get gridded hazard datasets
ds_tw = gridded_hazards.get("terrestrial_weather_gridded")
ds_wf = gridded_hazards.get("wildfire_gridded")
ds_sw = gridded_hazards.get("space_weather_gridded")

# Pick representative variables with actual data at this timestamp when possible
var_tw = _pick_var_with_data(ds_tw, VIS_TIMESTAMP, preferred="MAX_SHEAR") if ds_tw is not None else None
var_wf = _pick_var_with_data(ds_wf, VIS_TIMESTAMP, preferred="frp") if ds_wf is not None else None
var_sw = _pick_var_with_data(ds_sw, VIS_TIMESTAMP, preferred="Eh") if ds_sw is not None else None

print("Selected vars:", {"terrestrial_weather": var_tw, "wildfire": var_wf, "space_weather": var_sw})

# Three figures: one for each hazard
if ds_tw is not None and var_tw is not None:
    _plot_hazard_snapshot(ds_tw, var_tw, "Terrestrial Weather", VIS_TIMESTAMP, gdf_aoi_plot)
if ds_wf is not None and var_wf is not None:
    _plot_hazard_snapshot(ds_wf, var_wf, "Wildfire", VIS_TIMESTAMP, gdf_aoi_plot)
if ds_sw is not None and var_sw is not None:
    _plot_hazard_snapshot(ds_sw, var_sw, "Space Weather", VIS_TIMESTAMP, gdf_aoi_plot)

## Step 3: Ingest and Align Hazards

Read terrestrial weather, wildfire, and space-weather datasets and align them on the common grid and event timeline.

In [ ]:
ds_grid = xr.open_dataset(PATH_ANALYSIS_GRID)
print(ds_grid)

# Best-effort extraction of analysis-cell geometries if available in the dataset.
if "geometry" in ds_grid.variables:
    df_grid = ds_grid[["geometry"]].to_dataframe().reset_index()
    gdf_grid = gpd.GeoDataFrame(df_grid, geometry=gpd.GeoSeries.from_wkt(df_grid["geometry"]), crs="EPSG:4326")
    print(f"Grid cells parsed from geometry variable: {len(gdf_grid):,}")
else:
    gdf_grid = None
    print("No direct geometry variable found in grid dataset; gridding adapter cell will be added next.")

## Cross-Hazard Normalization: Variable Selection and Snapshot QA

This section implements a practical normalization scheme that respects mixed dimensionality:

- **Physical variables**: robust quantile scaling.
- **Derived physical variables**: robust quantile scaling (separate class).
- **Statistical/diagnostic variables**: special handling (ARI/probability/categorical).
- **Reflectivity (dBZ)**: threshold-aware scaling in dBZ space.
- **Wildfire and space weather**: robust quantile scaling with hazard-specific labels.

The next cells render **raw vs normalized** side-by-side for:
1. Space weather (e.g., `Eh`)
2. Terrestrial physical
3. Terrestrial derived physical
4. Terrestrial statistical/diagnostic
5. Wildfire

In [ ]:

# Timestamp used for side-by-side QA visualization
NORM_VIS_TIMESTAMP = pd.Timestamp("2023-03-30 20:00:00")


In [ ]:
# Normalization helpers and variable selection for mixed-dimensional terrestrial hazards
import re


def _clean_vals_for_plot(da):
    arr = np.asarray(da.values, dtype=float)
    fill_value = da.attrs.get("_FillValue", None)
    if fill_value is not None:
        arr[np.isclose(arr, float(fill_value))] = np.nan
    arr[arr <= -900] = np.nan
    return arr


def _nearest_time_da(ds, var_name, ts):
    da = ds[var_name].sel(time=pd.Timestamp(ts), method="nearest")
    used_time = pd.Timestamp(da["time"].values)
    return da, used_time


def _robust_quantile_scale(arr, qlo=0.05, qhi=0.95):
    out = np.full(arr.shape, np.nan, dtype=float)
    good = np.isfinite(arr)
    if not good.any():
        return out
    lo, hi = np.nanquantile(arr[good], [qlo, qhi])
    if np.isclose(lo, hi):
        lo, hi = np.nanmin(arr[good]), np.nanmax(arr[good])
    if np.isclose(lo, hi):
        out[good] = 0.5
        return out
    out[good] = np.clip((arr[good] - lo) / (hi - lo), 0.0, 1.0)
    return out


def _normalize_ari(arr):
    # ARI behaves like inverse probability. Use log10(ARI) then robust scaling.
    x = np.full(arr.shape, np.nan, dtype=float)
    good = np.isfinite(arr) & (arr > 0)
    if good.any():
        x[good] = np.log10(arr[good])
    return _robust_quantile_scale(x)


def _normalize_probability(arr):
    x = np.full(arr.shape, np.nan, dtype=float)
    good = np.isfinite(arr)
    if not good.any():
        return x
    vals = arr[good]
    # Handle either [0,1] or [0,100] representations.
    if np.nanmax(vals) > 1.5:
        vals = vals / 100.0
    x[good] = np.clip(vals, 0.0, 1.0)
    return x


def _normalize_categorical(arr):
    x = np.full(arr.shape, np.nan, dtype=float)
    good = np.isfinite(arr)
    if not good.any():
        return x
    vals = arr[good]
    u = np.unique(vals)
    if len(u) == 1:
        x[good] = 0.0 if u[0] <= 0 else 1.0
        return x
    rank_map = {v: i / (len(u) - 1) for i, v in enumerate(np.sort(u))}
    x[good] = np.array([rank_map[v] for v in vals], dtype=float)
    return x


def _normalize_dbz(arr):
    # Keep dBZ space; map to convective-relevance band.
    x = np.full(arr.shape, np.nan, dtype=float)
    good = np.isfinite(arr)
    if good.any():
        x[good] = np.clip((arr[good] - 20.0) / 40.0, 0.0, 1.0)
    return x


def _infer_var_class(var_name):
    n = var_name.lower()
    if "dbz" in n or "reflect" in n:
        return "reflectivity"
    if "ari" in n:
        return "stat_ari"
    if "prob" in n or "percent" in n:
        return "stat_prob"
    if "cat" in n or "class" in n or "flag" in n:
        return "stat_categorical"
    if "vil" in n or "vii" in n or "ivt" in n or "pwat" in n:
        return "derived_physical"
    return "physical"


def normalize_hazard_field(arr, var_name, family):
    cls = _infer_var_class(var_name)
    if cls == "reflectivity":
        return _normalize_dbz(arr), "reflectivity_dBZ_threshold_scaled"
    if cls == "stat_ari":
        return _normalize_ari(arr), "log10_ARI_robust_scaled"
    if cls == "stat_prob":
        return _normalize_probability(arr), "probability_clipped_0_1"
    if cls == "stat_categorical":
        return _normalize_categorical(arr), "categorical_rank_scaled"
    # For physical / derived / wildfire / space weather use robust quantile scaling.
    return _robust_quantile_scale(arr), f"{family}_robust_q05_q95"


def _choose_var_by_patterns(ds, include_patterns, exclude_patterns=None, ts=None):
    if ds is None:
        return None
    exclude_patterns = exclude_patterns or []
    vars_list = list(ds.data_vars)

    def _matches(v, pats):
        lv = v.lower()
        return any(re.search(p, lv) for p in pats)

    candidates = [v for v in vars_list if _matches(v, include_patterns) and not _matches(v, exclude_patterns)]
    if not candidates:
        return None

    if ts is None:
        return candidates[0]

    # Prefer candidate with finite data at timestamp.
    for v in candidates:
        da = ds[v].sel(time=pd.Timestamp(ts), method="nearest")
        arr = _clean_vals_for_plot(da)
        if np.isfinite(arr).any():
            return v
    return candidates[0]


# Pull hazard datasets from gridded bundle
if "gridded_hazards" not in globals():
    raise RuntimeError("gridded_hazards is not defined. Run hazard ingest/gridding cell first.")

ds_tw = gridded_hazards.get("terrestrial_weather_gridded")
ds_wf = gridded_hazards.get("wildfire_gridded")
ds_sw = gridded_hazards.get("space_weather_gridded")

# Variable picks by terrestrial class
var_tw_physical = _choose_var_by_patterns(
    ds_tw,
    include_patterns=[r"u10", r"v10", r"wspd", r"gust", r"tmp", r"temp", r"precip", r"rain"],
    exclude_patterns=[r"ari", r"prob", r"cat", r"vil", r"vii", r"dbz", r"reflect"],
    ts=NORM_VIS_TIMESTAMP,
)

var_tw_derived = _choose_var_by_patterns(
    ds_tw,
    include_patterns=[r"vil", r"vii", r"ivt", r"pwat"],
    exclude_patterns=[r"ari", r"prob", r"cat"],
    ts=NORM_VIS_TIMESTAMP,
)

var_tw_stat = _choose_var_by_patterns(
    ds_tw,
    include_patterns=[r"ari", r"prob", r"cat", r"class", r"flag"],
    exclude_patterns=[],
    ts=NORM_VIS_TIMESTAMP,
)

# If no stat variable exists, try reflectivity as a dedicated diagnostic proxy.
if var_tw_stat is None:
    var_tw_stat = _choose_var_by_patterns(
        ds_tw,
        include_patterns=[r"dbz", r"reflect"],
        exclude_patterns=[],
        ts=NORM_VIS_TIMESTAMP,
    )

var_sw = _choose_var_by_patterns(ds_sw, include_patterns=[r"eh", r"bh", r"geo", r"electric"], ts=NORM_VIS_TIMESTAMP)
if var_sw is None and ds_sw is not None and len(ds_sw.data_vars) > 0:
    var_sw = list(ds_sw.data_vars)[0]

var_wf = _choose_var_by_patterns(ds_wf, include_patterns=[r"frp", r"fire", r"radiative"], ts=NORM_VIS_TIMESTAMP)
if var_wf is None and ds_wf is not None and len(ds_wf.data_vars) > 0:
    var_wf = list(ds_wf.data_vars)[0]

print("Selected variables for normalization QA:")
print({
    "space_weather": var_sw,
    "terrestrial_physical": var_tw_physical,
    "terrestrial_derived": var_tw_derived,
    "terrestrial_stat_or_diag": var_tw_stat,
    "wildfire": var_wf,
})

In [ ]:
# Side-by-side snapshot: raw vs normalized for selected hazards

def _local_robust_limits(vals):
    good = vals[np.isfinite(vals)]
    if good.size == 0:
        return None, None
    vmin = np.nanpercentile(good, 2)
    vmax = np.nanpercentile(good, 98)
    if np.isclose(vmin, vmax):
        vmin = np.nanmin(good)
        vmax = np.nanmax(good)
    return float(vmin), float(vmax)


def _plot_raw_vs_normalized(ax_raw, ax_norm, ds, var_name, ts, title_prefix, family):
    if ds is None or var_name is None:
        ax_raw.axis("off")
        ax_norm.axis("off")
        ax_raw.text(0.5, 0.5, f"{title_prefix}: missing dataset/variable", ha="center", va="center")
        return

    da, used_time = _nearest_time_da(ds, var_name, ts)
    arr = _clean_vals_for_plot(da)

    if "lat" not in ds.coords or "lon" not in ds.coords:
        ax_raw.axis("off")
        ax_norm.axis("off")
        ax_raw.text(0.5, 0.5, f"{title_prefix}: missing lat/lon", ha="center", va="center")
        return

    lat2d = ds["lat"].values
    lon2d = ds["lon"].values
    flat_lat = lat2d.ravel()
    flat_lon = lon2d.ravel()
    flat_raw = arr.ravel()
    valid = np.isfinite(flat_raw)

    norm_arr, norm_label = normalize_hazard_field(arr, var_name, family)
    flat_norm = norm_arr.ravel()
    valid_norm = np.isfinite(flat_norm)

    if valid.any():
        vmin_raw, vmax_raw = _local_robust_limits(flat_raw)
        sc1 = ax_raw.scatter(
            flat_lon[valid], flat_lat[valid], c=flat_raw[valid], s=16, marker="s",
            linewidths=0, cmap="viridis", vmin=vmin_raw, vmax=vmax_raw
        )
        plt.colorbar(sc1, ax=ax_raw, fraction=0.045, pad=0.02, label=var_name)
    else:
        ax_raw.text(0.5, 0.5, "No valid raw data", ha="center", va="center", transform=ax_raw.transAxes)

    if valid_norm.any():
        sc2 = ax_norm.scatter(
            flat_lon[valid_norm], flat_lat[valid_norm], c=flat_norm[valid_norm], s=16, marker="s",
            linewidths=0, cmap="magma", vmin=0.0, vmax=1.0
        )
        plt.colorbar(sc2, ax=ax_norm, fraction=0.045, pad=0.02, label=f"normalized ({norm_label})")
    else:
        ax_norm.text(0.5, 0.5, "No valid normalized data", ha="center", va="center", transform=ax_norm.transAxes)

    for ax in [ax_raw, ax_norm]:
        if "gdf_aoi" in globals() and gdf_aoi is not None and not gdf_aoi.empty:
            gdf_aoi.to_crs("EPSG:4326").boundary.plot(ax=ax, color="black", linewidth=0.9)
        ax.set_xlim(-125, -65)
        ax.set_ylim(25, 50)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    ax_raw.set_title(f"{title_prefix} RAW | {var_name} | used={used_time}")
    ax_norm.set_title(f"{title_prefix} NORMALIZED | {var_name}")


rows = [
    (ds_sw, var_sw, "Space Weather", "space_weather"),
    (ds_tw, var_tw_physical, "Terrestrial Physical", "physical"),
    (ds_tw, var_tw_derived, "Terrestrial Derived", "derived_physical"),
    (ds_tw, var_tw_stat, "Terrestrial Stat/Diag", "stat_diag"),
    (ds_wf, var_wf, "Wildfire", "wildfire"),
]

nrows = len(rows)
fig, axes = plt.subplots(nrows=nrows, ncols=2, figsize=(16, 4 * nrows), constrained_layout=True)

if nrows == 1:
    axes = np.array([axes])

for i, (ds, var_name, title_prefix, family) in enumerate(rows):
    _plot_raw_vs_normalized(
        ax_raw=axes[i, 0],
        ax_norm=axes[i, 1],
        ds=ds,
        var_name=var_name,
        ts=NORM_VIS_TIMESTAMP,
        title_prefix=title_prefix,
        family=family,
    )

fig.suptitle(f"Raw vs Normalized Hazard Snapshot | requested={pd.Timestamp(NORM_VIS_TIMESTAMP)}", fontsize=14, y=1.01)
plt.show()

## Refinement QA: Nodata/Range Audit and Zero-Inflation-Aware Normalization

This section refines normalization by:

1. Auditing nodata patterns and effective value ranges per selected variable.
2. Applying a **zero-inflation-aware** transform for sparse hazards (especially precipitation-like and wildfire fields):
   - keep zero mass at 0,
   - normalize positive tail robustly,
   - optionally blend with positive exceedance rank.
3. Rendering a 3-way comparison for each hazard:
   - raw,
   - v1 normalized,
   - refined normalized.

In [ ]:
# Nodata/range diagnostics + refined normalization comparison (raw vs v1 vs refined)

def _array_diagnostics(arr):
    finite = np.isfinite(arr)
    n = arr.size
    n_finite = int(finite.sum())
    out = {
        "n_total": n,
        "n_finite": n_finite,
        "finite_frac": (n_finite / n) if n > 0 else np.nan,
        "zero_frac": np.nan,
        "q01": np.nan,
        "q50": np.nan,
        "q99": np.nan,
        "min": np.nan,
        "max": np.nan,
    }
    if n_finite > 0:
        vals = arr[finite]
        out["zero_frac"] = float(np.mean(np.isclose(vals, 0.0)))
        out["q01"], out["q50"], out["q99"] = [float(x) for x in np.nanquantile(vals, [0.01, 0.5, 0.99])]
        out["min"] = float(np.nanmin(vals))
        out["max"] = float(np.nanmax(vals))
    return out


def _zero_inflation_aware_scale(arr, qlo=0.05, qhi=0.95, blend_rank=0.35):
    """
    Refined scaling for sparse fields:
    - all non-positive values -> 0
    - positive values robustly scaled to [0,1]
    - blended with positive-only percentile rank to spread heavy tails
    """
    out = np.full(arr.shape, np.nan, dtype=float)
    finite = np.isfinite(arr)
    if not finite.any():
        return out

    vals = arr[finite]
    nonpos = vals <= 0
    pos = vals > 0

    scaled = np.zeros(vals.shape, dtype=float)
    if pos.any():
        pvals = vals[pos]
        lo, hi = np.nanquantile(pvals, [qlo, qhi])
        if np.isclose(lo, hi):
            lo, hi = np.nanmin(pvals), np.nanmax(pvals)
        if np.isclose(lo, hi):
            base = np.ones_like(pvals) * 0.5
        else:
            base = np.clip((pvals - lo) / (hi - lo), 0.0, 1.0)

        # Positive-only rank spread
        order = np.argsort(pvals)
        ranks = np.empty_like(order, dtype=float)
        ranks[order] = np.linspace(0.0, 1.0, len(pvals), endpoint=True)

        scaled[pos] = (1.0 - blend_rank) * base + blend_rank * ranks

    scaled[nonpos] = 0.0

    out_vals = np.full(vals.shape, np.nan, dtype=float)
    out_vals[:] = scaled
    out[finite] = out_vals
    return out


def normalize_hazard_field_refined(arr, var_name, family):
    cls = _infer_var_class(var_name)

    # Keep special cases
    if cls == "reflectivity":
        return _normalize_dbz(arr), "refined_reflectivity_dBZ_threshold"
    if cls == "stat_ari":
        # ARI as inverse-probability proxy; emphasize upper rarity structure in log-space
        x = np.full(arr.shape, np.nan, dtype=float)
        good = np.isfinite(arr) & (arr > 0)
        if good.any():
            x[good] = np.log10(arr[good])
        return _zero_inflation_aware_scale(x), "refined_log10_ARI_zero_inflation"
    if cls == "stat_prob":
        return _normalize_probability(arr), "refined_probability_0_1"
    if cls == "stat_categorical":
        return _normalize_categorical(arr), "refined_categorical_rank"

    # Physical/derived/wildfire/space: zero-aware scaling for sparse distributions
    return _zero_inflation_aware_scale(arr), f"refined_{family}_zero_inflation"


def _plot_raw_v1_refined(ax_raw, ax_v1, ax_ref, ds, var_name, ts, title_prefix, family):
    if ds is None or var_name is None:
        for ax in [ax_raw, ax_v1, ax_ref]:
            ax.axis("off")
        ax_raw.text(0.5, 0.5, f"{title_prefix}: missing dataset/variable", ha="center", va="center")
        return

    da, used_time = _nearest_time_da(ds, var_name, ts)
    arr = _clean_vals_for_plot(da)

    lat2d = ds["lat"].values
    lon2d = ds["lon"].values
    flat_lat = lat2d.ravel()
    flat_lon = lon2d.ravel()
    flat_raw = arr.ravel()
    valid_raw = np.isfinite(flat_raw)

    v1_arr, v1_label = normalize_hazard_field(arr, var_name, family)
    ref_arr, ref_label = normalize_hazard_field_refined(arr, var_name, family)
    flat_v1 = v1_arr.ravel()
    flat_ref = ref_arr.ravel()
    valid_v1 = np.isfinite(flat_v1)
    valid_ref = np.isfinite(flat_ref)

    # Raw
    if valid_raw.any():
        vmin_raw, vmax_raw = _local_robust_limits(flat_raw)
        sc_raw = ax_raw.scatter(flat_lon[valid_raw], flat_lat[valid_raw], c=flat_raw[valid_raw], s=14, marker="s",
                                linewidths=0, cmap="viridis", vmin=vmin_raw, vmax=vmax_raw)
        plt.colorbar(sc_raw, ax=ax_raw, fraction=0.045, pad=0.02, label=var_name)

    # V1 normalized
    if valid_v1.any():
        sc_v1 = ax_v1.scatter(flat_lon[valid_v1], flat_lat[valid_v1], c=flat_v1[valid_v1], s=14, marker="s",
                              linewidths=0, cmap="magma", vmin=0.0, vmax=1.0)
        plt.colorbar(sc_v1, ax=ax_v1, fraction=0.045, pad=0.02, label=v1_label)

    # Refined normalized
    if valid_ref.any():
        sc_ref = ax_ref.scatter(flat_lon[valid_ref], flat_lat[valid_ref], c=flat_ref[valid_ref], s=14, marker="s",
                                linewidths=0, cmap="magma", vmin=0.0, vmax=1.0)
        plt.colorbar(sc_ref, ax=ax_ref, fraction=0.045, pad=0.02, label=ref_label)

    for ax in [ax_raw, ax_v1, ax_ref]:
        if "gdf_aoi" in globals() and gdf_aoi is not None and not gdf_aoi.empty:
            gdf_aoi.to_crs("EPSG:4326").boundary.plot(ax=ax, color="black", linewidth=0.8)
        ax.set_xlim(-125, -65)
        ax.set_ylim(25, 50)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    ax_raw.set_title(f"{title_prefix} RAW | {var_name} | used={used_time}")
    ax_v1.set_title(f"{title_prefix} V1 NORMALIZED")
    ax_ref.set_title(f"{title_prefix} REFINED NORMALIZED")


rows_refine = [
    (ds_sw, var_sw, "Space Weather", "space_weather"),
    (ds_tw, var_tw_physical, "Terrestrial Physical", "physical"),
    (ds_tw, var_tw_derived, "Terrestrial Derived", "derived_physical"),
    (ds_tw, var_tw_stat, "Terrestrial Stat/Diag", "stat_diag"),
    (ds_wf, var_wf, "Wildfire", "wildfire"),
]

# Diagnostics table printout
print("Variable diagnostics at snapshot (raw values):")
for ds, var_name, title_prefix, _family in rows_refine:
    if ds is None or var_name is None:
        print(f"- {title_prefix}: missing")
        continue
    da, used_time = _nearest_time_da(ds, var_name, NORM_VIS_TIMESTAMP)
    arr = _clean_vals_for_plot(da)
    d = _array_diagnostics(arr)
    print(
        f"- {title_prefix} | {var_name} | used={used_time} | "
        f"finite={d['n_finite']}/{d['n_total']} ({d['finite_frac']:.2%}) | "
        f"zero_frac={d['zero_frac']:.2%} | q01={d['q01']:.3g} | q50={d['q50']:.3g} | q99={d['q99']:.3g}"
    )

# 3-way comparison figure
nrows = len(rows_refine)
fig, axes = plt.subplots(nrows=nrows, ncols=3, figsize=(21, 4 * nrows), constrained_layout=True)
if nrows == 1:
    axes = np.array([axes])

for i, (ds, var_name, title_prefix, family) in enumerate(rows_refine):
    _plot_raw_v1_refined(
        ax_raw=axes[i, 0],
        ax_v1=axes[i, 1],
        ax_ref=axes[i, 2],
        ds=ds,
        var_name=var_name,
        ts=NORM_VIS_TIMESTAMP,
        title_prefix=title_prefix,
        family=family,
    )

fig.suptitle(f"Hazard Normalization QA (Raw vs V1 vs Refined) | requested={pd.Timestamp(NORM_VIS_TIMESTAMP)}", fontsize=14, y=1.01)
plt.show()

## Step 3b: Hazard Variable Screening and Interaction Candidates

This section performs an initial, hazard-only screening pass (before Whisker/Eagle-I target coupling):

1. Group terrestrial variables by mechanism: wind loading, convective severity, precipitation/flooding, icing, thermal stress.
2. Compute robustness diagnostics per variable (coverage, zero inflation, temporal variability, spatial variability).
3. Build a first-pass ranking per mechanism group.
4. Construct and summarize key interaction candidates:
   - wildfire × wind,
   - two-or-more hazards above median at same place/time,
   - high wind × vegetation-impact proxy,
   - lightning × wildfire ignition proxy,
   - GIC preconditioning (space weather lead interactions).

In [ ]:
# Mechanism-based variable screening + interaction diagnostics (hazard-only first pass)
import re

if "gridded_hazards" not in globals():
    raise RuntimeError("gridded_hazards missing. Run hazard ingest cell first.")

# Ensure key handles exist
_ds_tw = gridded_hazards.get("terrestrial_weather_gridded")
_ds_wf = gridded_hazards.get("wildfire_gridded")
_ds_sw = gridded_hazards.get("space_weather_gridded")

if "NORM_VIS_TIMESTAMP" not in globals():
    NORM_VIS_TIMESTAMP = pd.Timestamp("2023-03-30 12:00:00")

# Local helpers (self-contained)
def _clean_vals(arr_like):
    arr = np.asarray(arr_like, dtype=float)
    arr[arr <= -900] = np.nan
    return arr

def _safe_sel(ds, var_name, ts):
    da = ds[var_name].sel(time=pd.Timestamp(ts), method="nearest")
    return da, pd.Timestamp(da["time"].values)

def _binary_above_median(arr):
    out = np.zeros(arr.shape, dtype=np.uint8)
    good = np.isfinite(arr)
    if not good.any():
        return out
    med = np.nanmedian(arr[good])
    out[good] = (arr[good] > med).astype(np.uint8)
    return out

def _screen_var(ds, var_name):
    da = ds[var_name]
    arr = _clean_vals(da.values)
    finite = np.isfinite(arr)
    if not finite.any():
        return {
            "var": var_name,
            "coverage": 0.0,
            "zero_frac": np.nan,
            "temporal_var": 0.0,
            "spatial_var": 0.0,
            "tail_ratio": np.nan,
            "score": 0.0,
        }

    vals = arr[finite]
    coverage = float(finite.mean())
    zero_frac = float(np.mean(np.isclose(vals, 0.0)))

    # Temporal and spatial variability metrics
    if arr.ndim == 3:  # time, y, x
        with np.errstate(invalid="ignore"):
            ts_mean = np.nanmean(arr, axis=(1, 2))
            sp_mean = np.nanmean(arr, axis=0)
        temporal_var = float(np.nanstd(ts_mean))
        spatial_var = float(np.nanstd(sp_mean))
    else:
        temporal_var = float(np.nanstd(vals))
        spatial_var = float(np.nanstd(vals))

    q50 = np.nanquantile(vals, 0.5)
    q95 = np.nanquantile(vals, 0.95)
    tail_ratio = float((q95 + 1e-9) / (abs(q50) + 1e-9))

    # Hazard-only screening score for first pass
    score = coverage * (1.0 - min(0.95, zero_frac)) * np.log1p(temporal_var + spatial_var) * np.log1p(max(0.0, tail_ratio - 1.0))

    return {
        "var": var_name,
        "coverage": coverage,
        "zero_frac": zero_frac,
        "temporal_var": temporal_var,
        "spatial_var": spatial_var,
        "tail_ratio": tail_ratio,
        "score": score,
    }


# Mechanism groups for terrestrial weather
MECH_GROUPS = {
    "wind_loading": [r"u10", r"v10", r"wspd", r"gust", r"wind"],
    "convective_severity": [r"shear", r"dbz", r"reflect", r"cape", r"vil", r"vii", r"hail", r"torn"],
    "precip_flooding": [r"precip", r"rain", r"qpe", r"flood", r"ari", r"pwat", r"ivt"],
    "icing": [r"ice", r"freez", r"frzr", r"sleet", r"snow"],
    "thermal": [r"temp", r"tmp", r"heat", r"cold", r"dew", r"rh"],
}

all_tw_vars = list(_ds_tw.data_vars) if _ds_tw is not None else []

def _match_any(name, pats):
    n = name.lower()
    return any(re.search(p, n) for p in pats)

group_rows = []
for grp, pats in MECH_GROUPS.items():
    grp_vars = [v for v in all_tw_vars if _match_any(v, pats)]
    for v in grp_vars:
        d = _screen_var(_ds_tw, v)
        d["group"] = grp
        group_rows.append(d)

var_screen_df = pd.DataFrame(group_rows)
if var_screen_df.empty:
    print("No terrestrial variables matched mechanism patterns.")
else:
    var_screen_df = var_screen_df.sort_values(["group", "score"], ascending=[True, False])
    print("Top 5 candidates per mechanism group (hazard-only screening):")
    for grp in MECH_GROUPS.keys():
        sub = var_screen_df[var_screen_df["group"] == grp].head(5)
        if len(sub) == 0:
            continue
        print(f"\n[{grp}]")
        print(sub[["var", "score", "coverage", "zero_frac", "tail_ratio"]].to_string(index=False))


# Pick representative variables for interaction diagnostics
# Wind variable preference: vector-consistent components first, then speed proxies
wind_var = None
for cand in ["u10", "v10", "wspd10", "wind_speed", "gust10"]:
    if cand in all_tw_vars:
        wind_var = cand
        break
if wind_var is None:
    wind_var = next((v for v in all_tw_vars if re.search(r"u10|v10|wspd|gust|wind", v.lower())), None)

lightning_var = next((v for v in all_tw_vars if re.search(r"lightning|ltg|flash", v.lower())), None)

space_var = var_sw if "var_sw" in globals() and var_sw in list(_ds_sw.data_vars) else (list(_ds_sw.data_vars)[0] if _ds_sw is not None and len(_ds_sw.data_vars) > 0 else None)
wildfire_var = var_wf if "var_wf" in globals() and var_wf in list(_ds_wf.data_vars) else (list(_ds_wf.data_vars)[0] if _ds_wf is not None and len(_ds_wf.data_vars) > 0 else None)

# One convective and one precip variable from screening
conv_var = None
precip_var = None
if "var_screen_df" in globals() and isinstance(var_screen_df, pd.DataFrame) and not var_screen_df.empty:
    conv_sub = var_screen_df[var_screen_df["group"] == "convective_severity"]
    precip_sub = var_screen_df[var_screen_df["group"] == "precip_flooding"]
    conv_var = conv_sub.iloc[0]["var"] if len(conv_sub) else None
    precip_var = precip_sub.iloc[0]["var"] if len(precip_sub) else None

print("\nInteraction variable choices:")
print({
    "wind_var": wind_var,
    "convective_var": conv_var,
    "precip_var": precip_var,
    "wildfire_var": wildfire_var,
    "lightning_var": lightning_var,
    "space_weather_var": space_var,
})

# Build binary exceedance masks at a consistent timestamp stack
def _to_binary_cube(ds, var_name):
    if ds is None or var_name is None:
        return None
    arr = _clean_vals(ds[var_name].values)
    return _binary_above_median(arr)

b_wind = _to_binary_cube(_ds_tw, wind_var)
b_conv = _to_binary_cube(_ds_tw, conv_var)
b_precip = _to_binary_cube(_ds_tw, precip_var)
b_wildfire = _to_binary_cube(_ds_wf, wildfire_var)
b_lightning = _to_binary_cube(_ds_tw, lightning_var)
b_space = _to_binary_cube(_ds_sw, space_var)

# Harmonize time dimension length for interactions across datasets
def _trim_time(arr, n):
    if arr is None:
        return None
    return arr[:n]

cubes = [x for x in [b_wind, b_conv, b_precip, b_wildfire, b_lightning, b_space] if x is not None]
if not cubes:
    raise RuntimeError("No interaction cubes available.")
ntime = min(c.shape[0] for c in cubes)

b_wind = _trim_time(b_wind, ntime)
b_conv = _trim_time(b_conv, ntime)
b_precip = _trim_time(b_precip, ntime)
b_wildfire = _trim_time(b_wildfire, ntime)
b_lightning = _trim_time(b_lightning, ntime)
b_space = _trim_time(b_space, ntime)

interaction_stats = []

def _rate(arr):
    if arr is None:
        return np.nan
    return float(np.mean(arr > 0))

# 1) Wildfire x wind
if b_wildfire is not None and b_wind is not None:
    inter = (b_wildfire & b_wind).astype(np.uint8)
    interaction_stats.append({"interaction": "wildfire_x_wind", "rate": _rate(inter)})

# 2) Any 2+ hazards above median (same time/place)
stack_for_combo = [x for x in [b_wind, b_conv, b_precip, b_wildfire, b_space] if x is not None]
if stack_for_combo:
    combo = (np.sum(np.stack(stack_for_combo, axis=0), axis=0) >= 2).astype(np.uint8)
    interaction_stats.append({"interaction": "two_or_more_hazards_above_median", "rate": _rate(combo)})

# 3) High wind x vegetation-impact proxy (using wildfire presence as proxy)
if b_wind is not None and b_wildfire is not None:
    veg_proxy = (b_wind & b_wildfire).astype(np.uint8)
    interaction_stats.append({"interaction": "high_wind_x_vegetation_proxy", "rate": _rate(veg_proxy)})

# 4) Lightning driving wildfire ignition (lead-lag proxy)
if b_lightning is not None and b_wildfire is not None:
    lags = [1, 2, 4]  # 1h, 2h, 4h if hourly data
    for lag in lags:
        if lag < ntime:
            lead = b_lightning[:-lag]
            resp = b_wildfire[lag:]
            cond = lead > 0
            p_resp = float(np.mean(resp[cond] > 0)) if np.any(cond) else np.nan
            base = float(np.mean(resp > 0))
            interaction_stats.append({"interaction": f"lightning_leads_wildfire_lag{lag}", "rate": p_resp, "baseline": base, "lift": (p_resp / base) if base > 0 else np.nan})

# 5) GIC preconditioning: space weather leading other hazards/disruption proxies
if b_space is not None:
    for target_name, target_cube in [("wind", b_wind), ("convective", b_conv), ("wildfire", b_wildfire)]:
        if target_cube is None:
            continue
        for lag in [1, 2, 4]:
            if lag < ntime:
                lead = b_space[:-lag]
                resp = target_cube[lag:]
                cond = lead > 0
                p_resp = float(np.mean(resp[cond] > 0)) if np.any(cond) else np.nan
                base = float(np.mean(resp > 0))
                interaction_stats.append({"interaction": f"spaceweather_leads_{target_name}_lag{lag}", "rate": p_resp, "baseline": base, "lift": (p_resp / base) if base > 0 else np.nan})

interaction_df = pd.DataFrame(interaction_stats)
print("\nInteraction summary:")
if interaction_df.empty:
    print("No interactions computed.")
else:
    show_cols = [c for c in ["interaction", "rate", "baseline", "lift"] if c in interaction_df.columns]
    print(interaction_df[show_cols].to_string(index=False))

# Optional hook for target-based scoring (if user injects merged panel later)
print("\nTarget-coupled variable scoring note:")
if "hazard_obs_panel" in globals():
    print("hazard_obs_panel detected: ready to run lead-lag AUC scoring in next cell.")
else:
    print("hazard_obs_panel not found yet. Next step: join hazard features with Whisker/Eagle-I targets and run AUC-based lead-lag screening.")

In [ ]:
# Step 3b refinement: enforce coverage-aware variable choices for interaction analysis
if "var_screen_df" not in globals() or var_screen_df.empty:
    raise RuntimeError("Run Step 3b screening cell first.")

def _pick_best(group_name, min_cov=0.5, min_score=0.0):
    sub = var_screen_df[var_screen_df["group"] == group_name].copy()
    if sub.empty:
        return None
    sub = sub[(sub["coverage"] >= min_cov) & (sub["score"] > min_score)]
    if sub.empty:
        return None
    return sub.sort_values("score", ascending=False).iloc[0]["var"]

wind_refined = _pick_best("wind_loading", min_cov=0.7, min_score=0.01)
conv_refined = _pick_best("convective_severity", min_cov=0.7, min_score=0.01)
precip_refined = _pick_best("precip_flooding", min_cov=0.7, min_score=0.01)

print("Coverage-aware picks:")
print({
    "wind": wind_refined,
    "convective": conv_refined,
    "precip": precip_refined,
})

print("\nTop convective candidates with nonzero score:")
conv_nonzero = var_screen_df[(var_screen_df["group"] == "convective_severity") & (var_screen_df["score"] > 0)].sort_values("score", ascending=False)
if conv_nonzero.empty:
    print("None found under current scoring. Consider alternate convective proxy (e.g., reflectivity percentile exceedance) in next pass.")
else:
    print(conv_nonzero[["var", "score", "coverage", "zero_frac", "tail_ratio"]].head(8).to_string(index=False))

## Step 4: Hazard to Stress Mapping (Grid to Components)

Project normalized hazard states from grid cells to transmission lines and substation elements to create component-level stress vectors.

In [ ]:
def build_substation_points(powergrid_gdf):
    """Build candidate substation points from line endpoints."""
    records = []
    for idx, geom in enumerate(powergrid_gdf.geometry):
        if geom is None:
            continue
        if geom.geom_type == "LineString":
            endpoints = [Point(geom.coords[0]), Point(geom.coords[-1])]
        elif geom.geom_type == "MultiLineString":
            endpoints = []
            for line in geom.geoms:
                endpoints.extend([Point(line.coords[0]), Point(line.coords[-1])])
        else:
            continue
        for p in endpoints:
            records.append({"line_idx": idx, "geometry": p})

    gdf_substations = gpd.GeoDataFrame(records, geometry="geometry", crs=powergrid_gdf.crs)
    return gdf_substations


def initialize_stress_vectors(powergrid_gdf, substations_gdf):
    """Initialize line/substation stress vectors for later hazard projection."""
    line_stress = pd.DataFrame({
        "line_idx": powergrid_gdf.index,
        "stress_state": 0.0,
    })
    substation_stress = pd.DataFrame({
        "substation_idx": substations_gdf.index,
        "stress_state": 0.0,
    })
    return line_stress, substation_stress


gdf_substations = build_substation_points(gdf_powergrid_aoi)
line_stress_df, substation_stress_df = initialize_stress_vectors(gdf_powergrid_aoi, gdf_substations)

print(f"Substation point candidates: {len(gdf_substations):,}")
print(f"Line stress vector length: {len(line_stress_df):,}")
print(f"Substation stress vector length: {len(substation_stress_df):,}")

## Step 5-7: Stress to Disruption to Spread Diagnostics

Placeholder for linking component stress to outage/disruption observations and evaluating propagation/spread across network and neighboring regions.

In [ ]:
def summarize_explanatory_support(line_stress_df, substation_stress_df, obs_paths):
    """Placeholder summary for explanatory support diagnostics."""
    summary = {
        "line_elements": int(len(line_stress_df)),
        "substation_elements": int(len(substation_stress_df)),
        "configured_observations": [k for k, v in obs_paths.items() if v is not None],
        "pending_observations": [k for k, v in obs_paths.items() if v is None],
    }
    return summary


support_summary = summarize_explanatory_support(line_stress_df, substation_stress_df, OBS_PATHS)
print("Event reconstruction scaffold summary:")
print(support_summary)

## Step 3c: Convective Proxy Builder + Impact Data Ingest

Build a robust convective hazard proxy from available terrestrial-weather channels, then ingest Whisker and Eagle-I impact datasets for hazard-to-impact coupling.

In [ ]:
# Convective proxy builder (coverage-aware, percentile-based)
import json

if "gridded_hazards" not in globals() or "terrestrial_weather_gridded" not in gridded_hazards:
    raise RuntimeError("Missing terrestrial weather dataset in gridded_hazards.")


ds_tw = gridded_hazards["terrestrial_weather_gridded"]
all_tw_vars = list(ds_tw.data_vars)


def _find_first(patterns, names):
    for p in patterns:
        for n in names:
            if re.search(p, n.lower()):
                return n
    return None


def _safe_standardize(da):
    arr = da.values.astype(float)
    arr[arr <= -900] = np.nan
    good = np.isfinite(arr)
    if not good.any():
        return xr.DataArray(np.zeros_like(arr, dtype=np.float32), dims=da.dims, coords=da.coords)
    q05, q95 = np.nanquantile(arr[good], [0.05, 0.95])
    if not np.isfinite(q95 - q05) or (q95 - q05) <= 1e-9:
        out = np.zeros_like(arr, dtype=np.float32)
    else:
        out = (arr - q05) / (q95 - q05)
        out = np.clip(out, 0.0, 1.0)
    out[~good] = np.nan
    return xr.DataArray(out.astype(np.float32), dims=da.dims, coords=da.coords)


# Candidate channels: reflectivity + shear + lightning
refl_var = _find_first([r"reflectivitycomposite", r"basereflectivity", r"max_reflect", r"reflect"], all_tw_vars)
shear_var = _find_first([r"shear"], all_tw_vars)
lightning_var = _find_first([r"lightning", r"ltg", r"flash"], all_tw_vars)

proxy_terms = []
weights = []

if refl_var is not None:
    proxy_terms.append(_safe_standardize(ds_tw[refl_var]))
    weights.append(0.50)
if shear_var is not None:
    proxy_terms.append(_safe_standardize(ds_tw[shear_var]))
    weights.append(0.30)
if lightning_var is not None:
    proxy_terms.append(_safe_standardize(ds_tw[lightning_var]))
    weights.append(0.20)

if not proxy_terms:
    raise RuntimeError("No convective proxy ingredients found (reflectivity/shear/lightning).")

w = np.array(weights, dtype=float)
w = w / w.sum()
conv_proxy = sum(float(wi) * term for wi, term in zip(w, proxy_terms))
conv_proxy.name = "convective_proxy"
conv_proxy.attrs["builder"] = "weighted_percentile_standardized_reflectivity_shear_lightning"
conv_proxy.attrs["source_vars"] = json.dumps({
    "reflectivity": refl_var,
    "shear": shear_var,
    "lightning": lightning_var,
})

# Persist back into hazard container
if "convective_proxy" in ds_tw.data_vars:
    ds_tw = ds_tw.drop_vars("convective_proxy")
ds_tw["convective_proxy"] = conv_proxy
gridded_hazards["terrestrial_weather_gridded"] = ds_tw

print("Convective proxy created.")
print({
    "reflectivity": refl_var,
    "shear": shear_var,
    "lightning": lightning_var,
    "weights": dict(zip([v for v in [refl_var, shear_var, lightning_var] if v is not None], list(w))),
})
print("Proxy coverage:", float(np.isfinite(conv_proxy.values).mean()))

In [ ]:
# Read Whisker + Eagle-I impacts and build hourly county panel (event window)
from pathlib import Path

WHISKER_DIR = Path("/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/Whisker_Labs_Data_March2026/2023-03-26 to 2023-04-09 Event")
EAGLE_DIR = Path("/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/outage_data/EAGLE-I")

# ---- Whisker outages ----
whisker_file = WHISKER_DIR / "PowerOutages.json"
if not whisker_file.exists():
    raise FileNotFoundError(f"Whisker file not found: {whisker_file}")

with open(whisker_file, "r", encoding="utf-8") as f:
    whisker_raw = json.load(f)

if isinstance(whisker_raw, dict) and "features" in whisker_raw:
    whisker_iter = whisker_raw["features"]
elif isinstance(whisker_raw, list):
    whisker_iter = whisker_raw
else:
    raise ValueError("Unexpected Whisker structure. Expected list or dict with 'features'.")

w_rows = []
for r in whisker_iter:
    loc = r.get("Location") or {}
    w_rows.append({
        "start_utc": pd.to_datetime(r.get("StartTime"), errors="coerce", utc=True),
        "start_local": pd.to_datetime(r.get("StartTimeLocal"), errors="coerce"),
        "sensors": pd.to_numeric(r.get("Sensors"), errors="coerce"),
        "average_amplitude": pd.to_numeric(r.get("AverageAmplitude"), errors="coerce"),
        "lat": pd.to_numeric(loc.get("Latitude"), errors="coerce"),
        "lon": pd.to_numeric(loc.get("Longitude"), errors="coerce"),
        "type": r.get("Type"),
    })

whisker_df = pd.DataFrame(w_rows)
whisker_df = whisker_df.dropna(subset=["start_utc", "lat", "lon"]).copy()
whisker_df = whisker_df[(whisker_df["start_utc"] >= EVENT_START) & (whisker_df["start_utc"] <= EVENT_END)].copy()
whisker_df["hour"] = whisker_df["start_utc"].dt.floor("h")
whisker_df["sensors"] = whisker_df["sensors"].fillna(1.0)

# Map points to county_fips via AOI county geometry if available
if "counties_aoi" in globals() and isinstance(counties_aoi, gpd.GeoDataFrame):
    county_ref = counties_aoi.copy()
elif "counties" in globals() and isinstance(counties, gpd.GeoDataFrame):
    county_ref = counties.copy()
else:
    county_ref = None

if county_ref is not None and len(county_ref) > 0:
    w_gdf = gpd.GeoDataFrame(
        whisker_df,
        geometry=gpd.points_from_xy(whisker_df["lon"], whisker_df["lat"]),
        crs="EPSG:4326",
    )
    county_ref = county_ref.to_crs("EPSG:4326")

    fips_col = next((c for c in ["county_fips", "GEOID", "FIPS", "fips"] if c in county_ref.columns), None)
    if fips_col is not None:
        joined = gpd.sjoin(w_gdf, county_ref[[fips_col, "geometry"]], how="left", predicate="within")
        whisker_df["county_fips"] = joined[fips_col].astype(str).str.zfill(5)
    else:
        whisker_df["county_fips"] = np.nan
else:
    whisker_df["county_fips"] = np.nan

whisker_hourly = (
    whisker_df
    .dropna(subset=["hour"]) 
    .groupby(["hour", "county_fips"], dropna=False)
    .agg(
        whisker_event_count=("type", "size"),
        whisker_sensors_sum=("sensors", "sum"),
    )
    .reset_index()
)
whisker_hourly["county_fips"] = whisker_hourly["county_fips"].replace({"nan": np.nan}).fillna("00000").astype(str).str.zfill(5)

# ---- Eagle-I county outages ----
eagle_candidates = sorted(EAGLE_DIR.glob("*2023*county_outage_data*.csv"))
if len(eagle_candidates) == 0:
    eagle_candidates = sorted(EAGLE_DIR.glob("*county_outage_data*.csv"))
if len(eagle_candidates) == 0:
    raise FileNotFoundError(f"No Eagle-I county outage CSV found in {EAGLE_DIR}")

eagle_file = eagle_candidates[0]
eaglei_data = pd.read_csv(eagle_file)

# Normalize Eagle-I columns across variants
col_map = {c.lower(): c for c in eaglei_data.columns}
run_col = col_map.get("run start time") or col_map.get("run_start_time")
fips_col = col_map.get("fips code") or col_map.get("fips_code")
out_col = col_map.get("customers out") or col_map.get("sum")
if run_col is None or fips_col is None or out_col is None:
    raise ValueError(f"Eagle-I required columns not found in {eagle_file.name}")

eaglei_data["run_start_time"] = pd.to_datetime(eaglei_data[run_col], errors="coerce", utc=True)
eaglei_data["fips_code"] = eaglei_data[fips_col].astype(str).str.zfill(5)
eaglei_data["customers_out"] = pd.to_numeric(eaglei_data[out_col], errors="coerce")
eaglei_data = eaglei_data.dropna(subset=["run_start_time", "fips_code"]).copy()
eaglei_data = eaglei_data[(eaglei_data["run_start_time"] >= EVENT_START) & (eaglei_data["run_start_time"] <= EVENT_END)].copy()
eaglei_data["hour"] = eaglei_data["run_start_time"].dt.floor("h")

eagle_hourly = (
    eaglei_data
    .groupby(["hour", "fips_code"], as_index=False)
    .agg(eagle_customers_out=("customers_out", "max"))
    .rename(columns={"fips_code": "county_fips"})
)
eagle_hourly["county_fips"] = eagle_hourly["county_fips"].astype(str).str.zfill(5)

# ---- Combined impact panel ----
impact_panel = pd.merge(
    eagle_hourly,
    whisker_hourly,
    on=["hour", "county_fips"],
    how="outer",
)

for c in ["eagle_customers_out", "whisker_event_count", "whisker_sensors_sum"]:
    if c in impact_panel.columns:
        impact_panel[c] = impact_panel[c].fillna(0)

print("Impact ingest summary")
print({
    "whisker_rows": int(len(whisker_df)),
    "whisker_hourly_rows": int(len(whisker_hourly)),
    "eagle_rows": int(len(eaglei_data)),
    "eagle_hourly_rows": int(len(eagle_hourly)),
    "impact_panel_rows": int(len(impact_panel)),
    "eagle_source": str(eagle_file),
})
print(impact_panel.head(10))

In [ ]:
# Interaction hotspot map with Whisker impact overlay
if "gridded_hazards" not in globals():
    raise RuntimeError("gridded_hazards missing. Run hazard ingest cells first.")
if "whisker_df" not in globals():
    raise RuntimeError("whisker_df missing. Run impact ingest cell first.")


ds_tw = gridded_hazards["terrestrial_weather_gridded"]
ds_wf = gridded_hazards.get("wildfire_gridded")

if "convective_proxy" not in ds_tw.data_vars:
    raise RuntimeError("convective_proxy not found. Run convective proxy builder cell first.")

# Choose wind variable, preferring a speed product if available
wind_var = None
for cand in ["wspd10", "wind_speed", "u10", "v10"]:
    if cand in ds_tw.data_vars:
        wind_var = cand
        break
if wind_var is None:
    raise RuntimeError("No wind variable available for hotspot interaction.")

wildfire_var = None
if ds_wf is not None and len(ds_wf.data_vars) > 0:
    wildfire_var = list(ds_wf.data_vars)[0]


def _clean_arr(a):
    a = np.asarray(a, dtype=float)
    a[a <= -900] = np.nan
    return a


def _binary_q80(a):
    out = np.zeros(a.shape, dtype=np.uint8)
    good = np.isfinite(a)
    if not good.any():
        return out
    thr = np.nanquantile(a[good], 0.80)
    out[good] = (a[good] >= thr).astype(np.uint8)
    return out


conv = _clean_arr(ds_tw["convective_proxy"].values)
wind = _clean_arr(ds_tw[wind_var].values)

b_conv = _binary_q80(conv)
b_wind = _binary_q80(wind)

cubes = [b_conv, b_wind]
labels = ["convective", "wind"]

if wildfire_var is not None:
    wf = _clean_arr(ds_wf[wildfire_var].values)
    b_wf = _binary_q80(wf)
    ntime = min(b_conv.shape[0], b_wind.shape[0], b_wf.shape[0])
    b_conv = b_conv[:ntime]
    b_wind = b_wind[:ntime]
    b_wf = b_wf[:ntime]
    cubes = [b_conv, b_wind, b_wf]
    labels = ["convective", "wind", "wildfire"]
else:
    ntime = min(b_conv.shape[0], b_wind.shape[0])
    b_conv = b_conv[:ntime]
    b_wind = b_wind[:ntime]

# Hotspot score: fraction of timesteps where 2+ hazards are concurrently high
combo = (np.sum(np.stack(cubes, axis=0), axis=0) >= 2).astype(np.uint8)
hotspot_rate = combo.mean(axis=0)

lat2d = ds_grid["lat"].values
lon2d = ds_grid["lon"].values

# Whisker overlay (recent event window points already filtered in whisker_df)
wplot = whisker_df.dropna(subset=["lat", "lon"]).copy()

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

mesh = ax.pcolormesh(
    lon2d,
    lat2d,
    hotspot_rate,
    shading="auto",
    cmap="YlOrRd",
    vmin=0,
    vmax=max(0.05, float(np.nanquantile(hotspot_rate[np.isfinite(hotspot_rate)], 0.99))),
)

if "gdf_aoi" in globals() and isinstance(gdf_aoi, gpd.GeoDataFrame):
    gdf_aoi.to_crs("EPSG:4326").boundary.plot(ax=ax, color="black", linewidth=0.6, alpha=0.8)

if "gdf_powergrid_aoi" in globals() and isinstance(gdf_powergrid_aoi, gpd.GeoDataFrame):
    gdf_powergrid_aoi.to_crs("EPSG:4326").plot(ax=ax, color="none", edgecolor="dimgray", linewidth=0.25, alpha=0.35)

if len(wplot) > 0:
    s = np.sqrt(wplot["sensors"].fillna(1).clip(lower=1).values) * 2.0
    ax.scatter(
        wplot["lon"],
        wplot["lat"],
        s=s,
        c="deepskyblue",
        alpha=0.35,
        edgecolor="none",
        label="Whisker outages (size~sensors)",
    )

cbar = fig.colorbar(mesh, ax=ax, shrink=0.8)
cbar.set_label("Hotspot rate (fraction of hours with 2+ high hazards)")

ax.set_title(
    "Hazard Interaction Hotspots with Whisker Outage Overlay\n"
    f"Inputs: {', '.join(labels)} | Wind var: {wind_var} | Wildfire var: {wildfire_var if wildfire_var else 'None'}"
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(loc="upper right", frameon=True)
ax.grid(alpha=0.15)
plt.tight_layout()
plt.show()

print({
    "hotspot_mean": float(np.nanmean(hotspot_rate)),
    "hotspot_p95": float(np.nanquantile(hotspot_rate[np.isfinite(hotspot_rate)], 0.95)),
    "whisker_points_overlayed": int(len(wplot)),
})

## Step 3d: Lagged Hazard-to-Impact Skill Scoring

Build a time-aligned hazard-feature table and score lead-lag predictive skill against impact targets from Eagle-I and Whisker. This provides a first-pass ranking of candidate predictors (including the convective proxy).

In [ ]:
# First-pass lagged predictor scoring against Eagle-I and Whisker impacts
from scipy.stats import spearmanr, rankdata

if "impact_panel" not in globals():
    raise RuntimeError("impact_panel is missing. Run the Step 3c impact ingest cell first.")
if "gridded_hazards" not in globals():
    raise RuntimeError("gridded_hazards is missing. Run hazard ingest cells first.")


def _as_utc_hour_index(idx_or_series):
    ts = pd.to_datetime(idx_or_series, errors="coerce", utc=True)
    return pd.DatetimeIndex(ts).floor("h")


def _clean_hazard_array(a):
    arr = np.asarray(a, dtype=float)
    arr[arr <= -900] = np.nan
    return arr


def _spatial_feature_timeseries(ds, var_name, q=0.95):
    if ds is None or var_name is None or var_name not in ds.data_vars:
        return None
    da = ds[var_name]
    arr = _clean_hazard_array(da.values)
    if arr.ndim != 3:
        return None

    with np.errstate(invalid="ignore"):
        mean_ts = np.nanmean(arr, axis=(1, 2))
        q_ts = np.nanquantile(arr, q, axis=(1, 2))

    t = _as_utc_hour_index(da["time"].values)
    out = pd.DataFrame({
        f"{var_name}__mean": mean_ts,
        f"{var_name}__q{int(q*100)}": q_ts,
    }, index=t)
    out = out[~out.index.duplicated(keep="first")].sort_index()
    return out


def _binary_auc(y_true, y_score):
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    ok = np.isfinite(y_score)
    y_true = y_true[ok]
    y_score = y_score[ok]
    if len(y_true) < 10:
        return np.nan
    n_pos = int((y_true == 1).sum())
    n_neg = int((y_true == 0).sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = rankdata(y_score)
    sum_pos = ranks[y_true == 1].sum()
    auc = (sum_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return float(auc)


# 1) Build impact targets at hourly resolution
impact_hourly = (
    impact_panel
    .copy()
    .assign(hour=lambda d: _as_utc_hour_index(d["hour"]))
    .groupby("hour", as_index=True)
    .agg(
        eagle_total_out=("eagle_customers_out", "sum"),
        whisker_event_count=("whisker_event_count", "sum"),
        whisker_sensors_sum=("whisker_sensors_sum", "sum"),
    )
    .sort_index()
)

impact_hourly["target_eagle_spike"] = (
    impact_hourly["eagle_total_out"] >= impact_hourly["eagle_total_out"].quantile(0.90)
).astype(int)
impact_hourly["target_whisker_spike"] = (
    impact_hourly["whisker_event_count"] >= impact_hourly["whisker_event_count"].quantile(0.90)
).astype(int)

# 2) Build predictor list (ensure convective proxy included)
ds_tw = gridded_hazards.get("terrestrial_weather_gridded")
ds_wf = gridded_hazards.get("wildfire_gridded")
ds_sw = gridded_hazards.get("space_weather_gridded")

predictor_specs = []

if ds_tw is not None and "convective_proxy" in ds_tw.data_vars:
    predictor_specs.append(("tw", "convective_proxy"))

for cand in ["wspd10", "u10", "v10", "MAX_SHEAR", "MergedReflectivityComposite_00.50", "MultiSensor_QPE_24H_Pass2_00.00"]:
    if ds_tw is not None and cand in ds_tw.data_vars and ("tw", cand) not in predictor_specs:
        predictor_specs.append(("tw", cand))

if ds_wf is not None:
    for cand in ["frp"]:
        if cand in ds_wf.data_vars and ("wf", cand) not in predictor_specs:
            predictor_specs.append(("wf", cand))
    if len(ds_wf.data_vars) > 0:
        fallback = list(ds_wf.data_vars)[0]
        if ("wf", fallback) not in predictor_specs:
            predictor_specs.append(("wf", fallback))

if ds_sw is not None:
    for cand in ["Bh", "Eh"]:
        if cand in ds_sw.data_vars and ("sw", cand) not in predictor_specs:
            predictor_specs.append(("sw", cand))
    if len(ds_sw.data_vars) > 0:
        fallback = list(ds_sw.data_vars)[0]
        if ("sw", fallback) not in predictor_specs:
            predictor_specs.append(("sw", fallback))

# 3) Build predictor dataframe
feature_frames = []
for fam, var in predictor_specs:
    ds = ds_tw if fam == "tw" else (ds_wf if fam == "wf" else ds_sw)
    feat = _spatial_feature_timeseries(ds, var, q=0.95)
    if feat is not None and len(feat) > 0:
        feature_frames.append(feat)

if not feature_frames:
    raise RuntimeError("No hazard predictor features could be built.")

hazard_hourly = pd.concat(feature_frames, axis=1).sort_index()

# 4) Align predictors + targets
model_df = hazard_hourly.join(impact_hourly, how="inner").sort_index()

# Keep finite predictors only for scoring
predictor_cols = [c for c in hazard_hourly.columns if c in model_df.columns]

if len(model_df) < 24:
    raise RuntimeError(f"Too few aligned hourly rows for scoring: {len(model_df)}")

# 5) Score lags
lags = list(range(0, 13))  # 0-12 hours lead
rows = []

for target_col in ["target_eagle_spike", "target_whisker_spike"]:
    for pcol in predictor_cols:
        x0 = model_df[pcol].astype(float)
        y0 = model_df[target_col].astype(int)

        for lag in lags:
            if lag == 0:
                x = x0.values
                y = y0.values
            else:
                x = x0.values[:-lag]
                y = y0.values[lag:]

            ok = np.isfinite(x) & np.isfinite(y)
            if ok.sum() < 20:
                continue

            xr = x[ok]
            yr = y[ok].astype(int)

            rho, _ = spearmanr(xr, yr)
            auc = _binary_auc(yr, xr)
            best_auc = np.nanmax([auc, 1.0 - auc]) if np.isfinite(auc) else np.nan

            rows.append({
                "target": target_col,
                "predictor": pcol,
                "lag_h": lag,
                "n": int(ok.sum()),
                "spearman_rho": float(rho) if np.isfinite(rho) else np.nan,
                "auc": auc,
                "best_auc": float(best_auc) if np.isfinite(best_auc) else np.nan,
            })

skill_long = pd.DataFrame(rows)
if skill_long.empty:
    raise RuntimeError("Skill table is empty. Check predictor/target overlap and finite values.")

# 6) Best lag summary per predictor/target
skill_best = (
    skill_long
    .sort_values(["target", "predictor", "best_auc", "spearman_rho"], ascending=[True, True, False, False])
    .groupby(["target", "predictor"], as_index=False)
    .first()
    .sort_values(["target", "best_auc", "spearman_rho"], ascending=[True, False, False])
)

print("Aligned rows for scoring:", len(model_df))
print("Predictor columns:", len(predictor_cols))
print("\nTop predictors for Eagle-I spike target:")
print(
    skill_best[skill_best["target"] == "target_eagle_spike"]
    [["predictor", "lag_h", "best_auc", "spearman_rho", "n"]]
    .head(12)
    .to_string(index=False)
)

print("\nTop predictors for Whisker spike target:")
print(
    skill_best[skill_best["target"] == "target_whisker_spike"]
    [["predictor", "lag_h", "best_auc", "spearman_rho", "n"]]
    .head(12)
    .to_string(index=False)
)

# 7) Quick visualization for top 6 predictors per target
for target_col in ["target_eagle_spike", "target_whisker_spike"]:
    top = skill_best[skill_best["target"] == target_col].head(6)
    if top.empty:
        continue

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(top["predictor"][::-1], top["best_auc"][::-1], color="teal", alpha=0.8)
    ax.set_xlim(0.5, 1.0)
    ax.set_xlabel("Best directional AUC across lags")
    ax.set_title(f"Top hazard predictors for {target_col}")
    plt.tight_layout()
    plt.show()

# Persist outputs for next cells
hazard_obs_skill_long = skill_long
hazard_obs_skill_best = skill_best
hazard_obs_model_df = model_df

## Step 3e: Multi-Hazard Event Chronology (With Spatial Modes)

Build a five-panel chronology that combines:
- Multi-hazard intensity proxy (convective + wildfire + space weather)
- Sag counts per hour (inside/outside CBEMA)
- Outage counts per hour
- Swell counts per hour
- Frequency-jump counts per hour

Spatial handling for hazard intensity is explicit via `SPATIAL_MODE`:
- `"conus_mean"`: equal-weight mean over all grid cells
- `"aoi_mean"`: mean over AOI grid cells only
- `"hotspot_weighted"`: weighted by persistent multi-hazard hotspot rate (preferred when focusing on impact-prone locations)

The cell also computes and overlays vertical markers for hazard onset, peak, and end.

In [ ]:
# Multi-hazard chronology with explicit spatial handling
from pathlib import Path

if "gridded_hazards" not in globals():
    raise RuntimeError("gridded_hazards is missing. Run hazard ingest cells first.")


def _as_utc_hour_index(x):
    ts = pd.to_datetime(x, errors="coerce", utc=True)
    return pd.DatetimeIndex(ts).floor("h")


def _clean_hazard_arr(a):
    arr = np.asarray(a, dtype=float)
    arr[arr <= -900] = np.nan
    return arr


def _robust_unit_scale(s):
    s = pd.Series(s, dtype=float)
    good = s.dropna()
    if good.empty:
        return s * np.nan
    q05, q95 = good.quantile([0.05, 0.95])
    if not np.isfinite(q95 - q05) or (q95 - q05) <= 1e-12:
        q05, q95 = float(good.min()), float(good.max())
    if not np.isfinite(q95 - q05) or (q95 - q05) <= 1e-12:
        return pd.Series(np.full(len(s), 0.5, dtype=float), index=s.index)
    out = (s - q05) / (q95 - q05)
    return out.clip(0.0, 1.0)


def _build_aoi_mask_2d(ds_ref, gdf_aoi_local):
    if ds_ref is None or "lat" not in ds_ref.coords or "lon" not in ds_ref.coords:
        return None
    if gdf_aoi_local is None or len(gdf_aoi_local) == 0:
        return None

    gdf_aoi_ll = gdf_aoi_local.to_crs("EPSG:4326")
    aoi_union = gdf_aoi_ll.unary_union

    lat2d = ds_ref["lat"].values
    lon2d = ds_ref["lon"].values
    ny, nx = lat2d.shape

    pts = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(lon2d.ravel(), lat2d.ravel()),
        crs="EPSG:4326",
    )
    mask_flat = pts.within(aoi_union).values
    return mask_flat.reshape(ny, nx)


def _timeseries_from_da(da, mode="conus_mean", mask2d=None, weights2d=None):
    if da is None:
        return None
    arr = _clean_hazard_arr(da.values)
    if arr.ndim != 3:
        return None

    if mode == "conus_mean":
        vals = np.nanmean(arr, axis=(1, 2))
    elif mode == "aoi_mean":
        if mask2d is None:
            vals = np.nanmean(arr, axis=(1, 2))
        else:
            vals = np.array([np.nanmean(arr[t][mask2d]) for t in range(arr.shape[0])], dtype=float)
    elif mode == "hotspot_weighted":
        if weights2d is None:
            vals = np.nanmean(arr, axis=(1, 2))
        else:
            w = np.asarray(weights2d, dtype=float)
            w = np.where(np.isfinite(w), w, 0.0)
            if np.nansum(w) <= 0:
                vals = np.nanmean(arr, axis=(1, 2))
            else:
                w = w / np.nansum(w)
                vals = np.array([np.nansum(arr[t] * w) for t in range(arr.shape[0])], dtype=float)
    else:
        raise ValueError(f"Unknown spatial mode: {mode}")

    idx = _as_utc_hour_index(da["time"].values)
    out = pd.Series(vals, index=idx)
    out = out[~out.index.duplicated(keep="first")].sort_index()
    return out


# ---- 1) Build multi-hazard proxy components ----
ds_tw = gridded_hazards.get("terrestrial_weather_gridded")
ds_wf = gridded_hazards.get("wildfire_gridded")
ds_sw = gridded_hazards.get("space_weather_gridded")

if ds_tw is None or "convective_proxy" not in ds_tw.data_vars:
    raise RuntimeError("convective_proxy missing. Run Step 3c convective proxy cell first.")

wildfire_var = None
if ds_wf is not None and len(ds_wf.data_vars) > 0:
    wildfire_var = "frp" if "frp" in ds_wf.data_vars else list(ds_wf.data_vars)[0]

space_var = None
if ds_sw is not None and len(ds_sw.data_vars) > 0:
    for cand in ["Eh", "Bh"]:
        if cand in ds_sw.data_vars:
            space_var = cand
            break
    if space_var is None:
        space_var = list(ds_sw.data_vars)[0]

# Spatial mode for Panel A
SPATIAL_MODE = "hotspot_weighted"  # options: conus_mean, aoi_mean, hotspot_weighted

# AOI mask for optional AOI-only aggregation
if "gdf_aoi" in globals() and isinstance(gdf_aoi, gpd.GeoDataFrame):
    aoi_mask2d = _build_aoi_mask_2d(ds_tw, gdf_aoi)
else:
    aoi_mask2d = None

# Hotspot weights for optional hotspot-weighted aggregation
# Build as fraction of hours where >=2 hazards exceed q80 at a grid cell.
def _binary_q80_cube(da):
    a = _clean_hazard_arr(da.values)
    good = np.isfinite(a)
    out = np.zeros(a.shape, dtype=np.uint8)
    if good.any():
        thr = np.nanquantile(a[good], 0.80)
        out[good] = (a[good] >= thr).astype(np.uint8)
    return out

cubes = []
cubes.append(_binary_q80_cube(ds_tw["convective_proxy"]))
if wildfire_var is not None:
    cubes.append(_binary_q80_cube(ds_wf[wildfire_var]))
if space_var is not None:
    cubes.append(_binary_q80_cube(ds_sw[space_var]))

ntime = min(c.shape[0] for c in cubes)
cubes = [c[:ntime] for c in cubes]
combo = (np.sum(np.stack(cubes, axis=0), axis=0) >= 2).astype(np.uint8)
hotspot_rate_2d = combo.mean(axis=0)
hotspot_weights_2d = np.where(np.isfinite(hotspot_rate_2d), hotspot_rate_2d, 0.0)
if np.nansum(hotspot_weights_2d) > 0:
    hotspot_weights_2d = hotspot_weights_2d / np.nansum(hotspot_weights_2d)
else:
    hotspot_weights_2d = None

conv_ts = _timeseries_from_da(
    ds_tw["convective_proxy"],
    mode=SPATIAL_MODE,
    mask2d=aoi_mask2d,
    weights2d=hotspot_weights_2d,
)
wf_ts = _timeseries_from_da(
    ds_wf[wildfire_var],
    mode=SPATIAL_MODE,
    mask2d=aoi_mask2d,
    weights2d=hotspot_weights_2d,
) if wildfire_var is not None else None
sw_ts = _timeseries_from_da(
    ds_sw[space_var],
    mode=SPATIAL_MODE,
    mask2d=aoi_mask2d,
    weights2d=hotspot_weights_2d,
) if space_var is not None else None

haz_df = pd.DataFrame(index=conv_ts.index)
haz_df["convective"] = conv_ts
if wf_ts is not None:
    haz_df["wildfire"] = wf_ts.reindex(haz_df.index)
if sw_ts is not None:
    haz_df["space_weather"] = sw_ts.reindex(haz_df.index)

for c in list(haz_df.columns):
    haz_df[f"{c}_norm"] = _robust_unit_scale(haz_df[c])

w_map = {
    "convective_norm": 0.50,
    "wildfire_norm": 0.25,
    "space_weather_norm": 0.25,
}
active_cols = [c for c in w_map if c in haz_df.columns]
w = np.array([w_map[c] for c in active_cols], dtype=float)
w = w / w.sum()
haz_df["multi_hazard_index"] = np.nansum(
    np.column_stack([haz_df[c].values for c in active_cols]) * w.reshape(1, -1),
    axis=1,
)

# Keep only event window and hourly UTC index
event_hours = pd.date_range(EVENT_START.floor("h"), EVENT_END.floor("h"), freq="1h", tz="UTC")
haz_df = haz_df.reindex(event_hours)


# ---- 2) Build hourly Whisker event channels (sag/swell/frequency/frequency jump) ----
WHISKER_DIR = Path(
    "/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/Whisker_Labs_Data_March2026/2023-03-26 to 2023-04-09 Event"
)

event_file_map = {
    "Frequency": "Frequency.json",
    "Sag_Inside_CBEMA": "Sags_Inside_CBEMA.json",
    "Sag_Outside_CBEMA": "Sags_Outside_CBEMA.json",
    "Swell_Inside_CBEMA": "Swells_Inside_CBEMA.json",
    "Swell_Outside_CBEMA": "Swells_Outside_CBEMA.json",
}

rows = []
for group_name, fn in event_file_map.items():
    fp = WHISKER_DIR / fn
    if not fp.exists():
        continue
    with open(fp, "r", encoding="utf-8") as f:
        raw = json.load(f)

    if isinstance(raw, dict) and "features" in raw:
        it = raw["features"]
    elif isinstance(raw, list):
        it = raw
    else:
        continue

    for r in it:
        rows.append({
            "event_group": group_name,
            "event_type": r.get("Type", group_name),
            "start_utc": pd.to_datetime(r.get("StartTime"), errors="coerce", utc=True),
            "average_amplitude": pd.to_numeric(r.get("AverageAmplitude"), errors="coerce"),
        })

events = pd.DataFrame(rows)
if events.empty:
    raise RuntimeError("No Whisker event files loaded for chronology panels.")

events = events.dropna(subset=["start_utc"]).copy()
events = events[(events["start_utc"] >= EVENT_START) & (events["start_utc"] <= EVENT_END)].copy()
events["hour"] = events["start_utc"].dt.floor("h")

chron = pd.DataFrame(index=event_hours)
chron["sag_inside_count"] = (
    events[events["event_group"] == "Sag_Inside_CBEMA"]
    .groupby("hour").size().reindex(event_hours, fill_value=0)
)
chron["sag_outside_count"] = (
    events[events["event_group"] == "Sag_Outside_CBEMA"]
    .groupby("hour").size().reindex(event_hours, fill_value=0)
)
chron["swell_total_count"] = (
    events[events["event_group"].isin(["Swell_Inside_CBEMA", "Swell_Outside_CBEMA"])]
    .groupby("hour").size().reindex(event_hours, fill_value=0)
)

freq_mask = events["event_group"] == "Frequency"
freq_jump_mask = freq_mask & events["event_type"].astype(str).str.contains("jump", case=False, na=False)
if int(freq_jump_mask.sum()) == 0:
    # Fallback when type tags are absent
    freq_jump_mask = freq_mask

chron["frequency_jump_count"] = (
    events[freq_jump_mask]
    .groupby("hour").size().reindex(event_hours, fill_value=0)
)


# ---- 3) Outage count per hour (prefer Eagle-I county count > 0) ----
if "eaglei_data" in globals() and isinstance(eaglei_data, pd.DataFrame) and len(eaglei_data) > 0:
    e = eaglei_data.copy()
    e["hour"] = _as_utc_hour_index(e["run_start_time"])
    if "customers_out" in e.columns:
        e = e[np.isfinite(pd.to_numeric(e["customers_out"], errors="coerce"))].copy()
        e["customers_out"] = pd.to_numeric(e["customers_out"], errors="coerce")
        out_ct = e.groupby("hour").apply(lambda d: int((d["customers_out"] > 0).sum()))
    else:
        out_ct = e.groupby("hour").size().astype(int)
    chron["outage_count"] = out_ct.reindex(event_hours, fill_value=0)
elif "impact_panel" in globals() and isinstance(impact_panel, pd.DataFrame) and len(impact_panel) > 0:
    ip = impact_panel.copy()
    ip["hour"] = _as_utc_hour_index(ip["hour"])
    if "eagle_customers_out" in ip.columns:
        tmp = (
            ip.groupby(["hour", "county_fips"], as_index=False)
            .agg(eagle_customers_out=("eagle_customers_out", "max"))
        )
        out_ct = tmp.groupby("hour").apply(lambda d: int((pd.to_numeric(d["eagle_customers_out"], errors="coerce") > 0).sum()))
        chron["outage_count"] = out_ct.reindex(event_hours, fill_value=0)
    else:
        chron["outage_count"] = 0
else:
    # Fallback to Whisker power outage events
    power_fp = WHISKER_DIR / "PowerOutages.json"
    if power_fp.exists():
        with open(power_fp, "r", encoding="utf-8") as f:
            prow = json.load(f)
        pit = prow["features"] if isinstance(prow, dict) and "features" in prow else (prow if isinstance(prow, list) else [])
        p_df = pd.DataFrame({
            "start_utc": [pd.to_datetime(r.get("StartTime"), errors="coerce", utc=True) for r in pit]
        }).dropna()
        p_df = p_df[(p_df["start_utc"] >= EVENT_START) & (p_df["start_utc"] <= EVENT_END)].copy()
        p_df["hour"] = p_df["start_utc"].dt.floor("h")
        chron["outage_count"] = p_df.groupby("hour").size().reindex(event_hours, fill_value=0)
    else:
        chron["outage_count"] = 0


# ---- 4) Merge with hazard index and compute markers ----
panel = chron.join(haz_df[["multi_hazard_index"] + active_cols], how="left")
panel["multi_hazard_index"] = panel["multi_hazard_index"].interpolate(limit_direction="both")

for c in ["sag_inside_count", "sag_outside_count", "swell_total_count", "outage_count", "frequency_jump_count"]:
    panel[c] = pd.to_numeric(panel[c], errors="coerce").fillna(0.0)

mh = panel["multi_hazard_index"].copy()
if mh.isna().all():
    raise RuntimeError("Multi-hazard index is all-NaN; check hazard datasets and spatial mode.")

thr_onset = float(mh.quantile(0.75))
thr_end = float(mh.quantile(0.50))

onset_candidates = mh[mh >= thr_onset]
onset_time = onset_candidates.index[0] if len(onset_candidates) else mh.index[0]
peak_time = mh.idxmax()
post_peak = mh.loc[peak_time:]
end_candidates = post_peak[post_peak <= thr_end]
end_time = end_candidates.index[0] if len(end_candidates) else post_peak.index[-1]


# ---- 5) Plot event chronology Panels A-E ----
fig, axes = plt.subplots(5, 1, figsize=(16, 14), sharex=True, constrained_layout=True)

# Panel A: multi-hazard intensity
axes[0].plot(panel.index, panel["multi_hazard_index"], color="tab:orange", linewidth=2.0, label="Multi-hazard index")
for c in active_cols:
    axes[0].plot(panel.index, panel[c], linewidth=1.0, alpha=0.65, label=c.replace("_norm", ""))
axes[0].set_ylabel("Index (0-1)")
axes[0].set_title(f"Panel A: Multi-hazard intensity ({SPATIAL_MODE})")
axes[0].legend(loc="upper left", ncol=2, fontsize=9)
axes[0].grid(alpha=0.2)

# Panel B: sag inside/outside CBEMA
axes[1].plot(panel.index, panel["sag_inside_count"], linewidth=1.4, label="Sag inside CBEMA")
axes[1].plot(panel.index, panel["sag_outside_count"], linewidth=1.4, label="Sag outside CBEMA")
axes[1].set_ylabel("Count/hr")
axes[1].set_title("Panel B: Sag counts per hour")
axes[1].legend(loc="upper left")
axes[1].grid(alpha=0.2)

# Panel C: outage counts per hour
axes[2].plot(panel.index, panel["outage_count"], color="black", linewidth=1.6, label="Outage count")
axes[2].set_ylabel("Count/hr")
axes[2].set_title("Panel C: Outage counts per hour")
axes[2].legend(loc="upper left")
axes[2].grid(alpha=0.2)

# Panel D: swell counts per hour
axes[3].plot(panel.index, panel["swell_total_count"], color="tab:purple", linewidth=1.6, label="Swell total")
axes[3].set_ylabel("Count/hr")
axes[3].set_title("Panel D: Swell counts per hour")
axes[3].legend(loc="upper left")
axes[3].grid(alpha=0.2)

# Panel E: frequency jump counts per hour
axes[4].plot(panel.index, panel["frequency_jump_count"], color="tab:green", linewidth=1.6, label="Frequency jump")
axes[4].set_ylabel("Count/hr")
axes[4].set_title("Panel E: Frequency-jump counts per hour")
axes[4].legend(loc="upper left")
axes[4].grid(alpha=0.2)

for i, ax in enumerate(axes):
    ax.axvline(onset_time, color="tab:blue", linestyle="--", linewidth=1.2, label="Hazard onset" if i == 0 else None)
    ax.axvline(peak_time, color="tab:red", linestyle="--", linewidth=1.2, label="Hazard peak" if i == 0 else None)
    ax.axvline(end_time, color="tab:gray", linestyle="--", linewidth=1.2, label="Hazard end" if i == 0 else None)

axes[-1].set_xlabel("UTC time")
fig.suptitle("Event Chronology: Multi-Hazard Intensity and Power-Quality/Outage Responses", fontsize=14)
plt.show()

print("Chronology markers:")
print({
    "spatial_mode": SPATIAL_MODE,
    "onset": str(onset_time),
    "peak": str(peak_time),
    "end": str(end_time),
    "convective_var": "convective_proxy",
    "wildfire_var": wildfire_var,
    "space_weather_var": space_var,
})

# Persist output for reuse
event_chronology_panel = panel
event_hazard_markers = {
    "onset": onset_time,
    "peak": peak_time,
    "end": end_time,
    "spatial_mode": SPATIAL_MODE,
}

## Step 3f: Fine-Area Event Chronology (Utility -> County -> State)

Build a stratified chronology to preserve local signal and reduce aggregation noise.

This cell:
- Loads utility service territories.
- Computes hazard chronology by utility/county/state from gridded hazards.
- Joins Whisker and Eagle-I impacts at matching geographies.
- Produces comparable hourly panels for local model calibration and validation.

In [ ]:
from pathlib import Path

UTILITY_BOUNDARY_PATH = Path("/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/location_data/Electric_Utilities_Data/Electric_Retail_Service_Territories/Retail_Service_Territories.shp")
COUNTY_BOUNDARY_PATH = Path("/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/location_data/counties.geojson")
PATH_AOI = Path("data/ca_and_bordering_states.geojson")

if "gridded_hazards" not in globals():
    raise RuntimeError("gridded_hazards is missing. Run hazard ingest cells first.")
if "EVENT_START" not in globals() or "EVENT_END" not in globals():
    raise RuntimeError("EVENT_START / EVENT_END are missing.")


def _as_hour_utc(x):
    return pd.DatetimeIndex(pd.to_datetime(x, errors="coerce", utc=True)).floor("h")


def _pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def _clean_cube(arr):
    out = np.asarray(arr, dtype=float)
    out[out <= -900] = np.nan
    return out


def _seasonal_exceedance(series):
    s = pd.Series(series, dtype=float)
    if s.isna().all():
        return s
    return s.groupby(s.index.month).transform(lambda x: x.rank(pct=True, method="average"))


def _find_first(names, patterns):
    lower = [n.lower() for n in names]
    for pat in patterns:
        for i, n in enumerate(lower):
            if re.search(pat, n):
                return names[i]
    return None


# ---- Geometry inputs ----
if not UTILITY_BOUNDARY_PATH.exists():
    raise FileNotFoundError(f"Utility boundary file not found: {UTILITY_BOUNDARY_PATH}")
if not COUNTY_BOUNDARY_PATH.exists():
    raise FileNotFoundError(f"County boundary file not found: {COUNTY_BOUNDARY_PATH}")

utility_gdf = gpd.read_file(UTILITY_BOUNDARY_PATH).to_crs("EPSG:4326")
counties_gdf = gpd.read_file(COUNTY_BOUNDARY_PATH).to_crs("EPSG:4326")

# Preserve both a stable ID and a human-readable name
util_id_col = _pick_col(
    utility_gdf,
    ["id", "ID", "objectid", "OBJECTID", "utility_id"],
)
util_name_col = _pick_col(
    utility_gdf,
    ["name", "NAME", "UtilityName", "utilityname", "UTILITY_NAME", "Company", "company", "Provider", "provider"],
)

if util_id_col is None:
    utility_gdf["utility_id"] = [f"utility_{i}" for i in range(len(utility_gdf))]
else:
    utility_gdf["utility_id"] = utility_gdf[util_id_col].astype(str)

if util_name_col is None:
    utility_gdf["utility_name"] = utility_gdf["utility_id"].astype(str)
else:
    utility_gdf["utility_name"] = utility_gdf[util_name_col].astype(str)

utility_gdf["utility_id"] = utility_gdf["utility_id"].fillna("unknown_utility")
utility_gdf["utility_name"] = utility_gdf["utility_name"].fillna(utility_gdf["utility_id"])
utility_gdf = utility_gdf[["utility_id", "utility_name", "geometry"]].copy()

# Clip counties to the AOI for local-area chronology and propagation joins
if "gdf_aoi" in globals() and isinstance(gdf_aoi, gpd.GeoDataFrame) and len(gdf_aoi) > 0:
    gdf_aoi_local = gdf_aoi.to_crs("EPSG:4326")
    counties_gdf = gpd.clip(counties_gdf, gdf_aoi_local).reset_index(drop=True)
else:
    gdf_aoi_local = None

county_fips_col = _pick_col(counties_gdf, ["county_fips", "GEOID", "COUNTYFP", "FIPS", "fips"])
county_name_col = _pick_col(counties_gdf, ["NAME", "name", "CountyName", "COUNTY"])

if county_fips_col is not None:
    counties_gdf["county_fips"] = counties_gdf[county_fips_col].astype(str).str.zfill(5)
else:
    counties_gdf["county_fips"] = [f"county_{i}" for i in range(len(counties_gdf))]

if county_name_col is not None:
    counties_gdf["county_name"] = counties_gdf[county_name_col].astype(str)
else:
    counties_gdf["county_name"] = counties_gdf["county_fips"].astype(str)

counties_gdf = counties_gdf[["county_fips", "county_name", "geometry"]].copy()
county_gdf = counties_gdf.copy()

if PATH_AOI.exists():
    state_gdf = gpd.read_file(PATH_AOI).to_crs("EPSG:4326")
    state_id_col = _pick_col(state_gdf, ["STUSPS", "STATE_ABBR", "STATE", "NAME", "STATE_NAME", "state"])
    if state_id_col is not None:
        state_gdf["state_id"] = state_gdf[state_id_col].astype(str)
    else:
        state_gdf["state_id"] = [f"state_{i}" for i in range(len(state_gdf))]
    state_gdf = state_gdf[["state_id", "geometry"]].copy()
else:
    state_gdf = None


# ---- Hazard ingredients ----
ds_tw = gridded_hazards.get("terrestrial_weather_gridded")
ds_wf = gridded_hazards.get("wildfire_gridded")
ds_sw = gridded_hazards.get("space_weather_gridded")

if ds_tw is None:
    raise RuntimeError("terrestrial_weather_gridded missing in gridded_hazards")

conv_var = "convective_proxy" if "convective_proxy" in ds_tw.data_vars else _find_first(list(ds_tw.data_vars), [r"reflect", r"shear", r"storm"])
if conv_var is None:
    raise RuntimeError("No convective proxy/variable found for chronology build.")

wf_var = None
if ds_wf is not None and len(ds_wf.data_vars) > 0:
    wf_var = "frp" if "frp" in ds_wf.data_vars else list(ds_wf.data_vars)[0]

sw_var = None
if ds_sw is not None and len(ds_sw.data_vars) > 0:
    sw_var = "Eh" if "Eh" in ds_sw.data_vars else ("Bh" if "Bh" in ds_sw.data_vars else list(ds_sw.data_vars)[0])

conv_cube = _clean_cube(ds_tw[conv_var].values)
ntime = conv_cube.shape[0]
lat2d = ds_tw["lat"].values
lon2d = ds_tw["lon"].values

wf_cube = _clean_cube(ds_wf[wf_var].values[:ntime]) if wf_var is not None else None
sw_cube = _clean_cube(ds_sw[sw_var].values[:ntime]) if sw_var is not None else None

if wf_cube is not None:
    ntime = min(ntime, wf_cube.shape[0])
if sw_cube is not None:
    ntime = min(ntime, sw_cube.shape[0])

conv_cube = conv_cube[:ntime]
if wf_cube is not None:
    wf_cube = wf_cube[:ntime]
if sw_cube is not None:
    sw_cube = sw_cube[:ntime]

haz_time = _as_hour_utc(ds_tw["time"].values[:ntime])
event_hours = pd.date_range(EVENT_START.floor("h"), EVENT_END.floor("h"), freq="1h", tz="UTC")


# ---- Grid-cell to region mapping ----
grid_pts = gpd.GeoDataFrame(
    {"grid_idx": np.arange(lat2d.size, dtype=int)},
    geometry=gpd.points_from_xy(lon2d.ravel(), lat2d.ravel()),
    crs="EPSG:4326",
)


def _map_grid_to_region(points_gdf, region_gdf, key):
    if region_gdf is None or len(region_gdf) == 0:
        return None
    j = gpd.sjoin(points_gdf, region_gdf[[key, "geometry"]], how="left", predicate="within")
    out = j[["grid_idx", key]].copy().dropna(subset=[key])
    out[key] = out[key].astype(str)
    return out


map_utility = _map_grid_to_region(grid_pts, utility_gdf, "utility_id")
map_county = _map_grid_to_region(grid_pts, county_gdf, "county_fips") if county_gdf is not None else None
map_state = _map_grid_to_region(grid_pts, state_gdf, "state_id") if state_gdf is not None else None


def _aggregate_cube_by_map(cube, map_df, key):
    if cube is None or map_df is None or len(map_df) == 0:
        return None
    arr = cube.reshape(cube.shape[0], -1)
    idx = map_df["grid_idx"].values.astype(int)
    grp = map_df[key].values.astype(str)

    rows = []
    for t in range(arr.shape[0]):
        dft = pd.DataFrame({"region": grp, "val": arr[t, idx]})
        g = dft.groupby("region", as_index=False)["val"].mean()
        g["hour"] = haz_time[t]
        rows.append(g)
    return pd.concat(rows, ignore_index=True).rename(columns={"region": key, "val": "value"})


haz_util_conv = _aggregate_cube_by_map(conv_cube, map_utility, "utility_id")
haz_county_conv = _aggregate_cube_by_map(conv_cube, map_county, "county_fips") if map_county is not None else None
haz_state_conv = _aggregate_cube_by_map(conv_cube, map_state, "state_id") if map_state is not None else None

haz_util_wf = _aggregate_cube_by_map(wf_cube, map_utility, "utility_id") if wf_cube is not None else None
haz_county_wf = _aggregate_cube_by_map(wf_cube, map_county, "county_fips") if (wf_cube is not None and map_county is not None) else None
haz_state_wf = _aggregate_cube_by_map(wf_cube, map_state, "state_id") if (wf_cube is not None and map_state is not None) else None

haz_util_sw = _aggregate_cube_by_map(sw_cube, map_utility, "utility_id") if sw_cube is not None else None
haz_county_sw = _aggregate_cube_by_map(sw_cube, map_county, "county_fips") if (sw_cube is not None and map_county is not None) else None
haz_state_sw = _aggregate_cube_by_map(sw_cube, map_state, "state_id") if (sw_cube is not None and map_state is not None) else None


def _build_hazard_index(df_conv, df_wf, df_sw, key):
    blocks = []
    if df_conv is not None:
        blocks.append(df_conv.rename(columns={"value": "conv_raw"})[["hour", key, "conv_raw"]])
    if df_wf is not None:
        blocks.append(df_wf.rename(columns={"value": "wf_raw"})[["hour", key, "wf_raw"]])
    if df_sw is not None:
        blocks.append(df_sw.rename(columns={"value": "sw_raw"})[["hour", key, "sw_raw"]])
    if not blocks:
        return None

    out = blocks[0]
    for b in blocks[1:]:
        out = out.merge(b, on=["hour", key], how="outer")
    out = out.sort_values([key, "hour"]).copy()

    for c in ["conv_raw", "wf_raw", "sw_raw"]:
        if c in out.columns:
            exc = pd.Series(np.nan, index=out.index, dtype=float)
            for _, d in out.groupby(key):
                s = pd.Series(d[c].values, index=pd.DatetimeIndex(d["hour"]))
                exc.loc[d.index] = _seasonal_exceedance(s).values
            out[f"{c}_exc"] = exc.values

    w = {"conv_raw_exc": 0.50, "wf_raw_exc": 0.25, "sw_raw_exc": 0.25}
    cols = [c for c in w if c in out.columns]
    if not cols:
        out["hazard_index"] = np.nan
    else:
        ww = np.array([w[c] for c in cols], dtype=float)
        ww = ww / ww.sum()
        out["hazard_index"] = np.nansum(np.column_stack([out[c].values for c in cols]) * ww.reshape(1, -1), axis=1)
    return out


haz_utility = _build_hazard_index(haz_util_conv, haz_util_wf, haz_util_sw, "utility_id")
haz_county = _build_hazard_index(haz_county_conv, haz_county_wf, haz_county_sw, "county_fips") if haz_county_conv is not None else None
haz_state = _build_hazard_index(haz_state_conv, haz_state_wf, haz_state_sw, "state_id") if haz_state_conv is not None else None

# Attach utility_name to hazard_utility for easier plotting/inspection
if haz_utility is not None:
    util_name_ref = utility_gdf[["utility_id", "utility_name"]].drop_duplicates()
    haz_utility = haz_utility.merge(util_name_ref, on="utility_id", how="left")


# ---- Impact chronology by geography ----
whisker_local = whisker_df.copy() if "whisker_df" in globals() else pd.DataFrame()
if len(whisker_local) > 0:
    whisker_local["hour"] = _as_hour_utc(whisker_local["start_utc"]) if "start_utc" in whisker_local.columns else _as_hour_utc(whisker_local["hour"])
    if "sensors" not in whisker_local.columns:
        whisker_local["sensors"] = 1.0
    if "lat" in whisker_local.columns and "lon" in whisker_local.columns:
        wg = gpd.GeoDataFrame(whisker_local, geometry=gpd.points_from_xy(whisker_local["lon"], whisker_local["lat"]), crs="EPSG:4326")
    else:
        wg = None
else:
    wg = None

if wg is not None:
    w_util = gpd.sjoin(wg, utility_gdf[["utility_id", "geometry"]], how="left", predicate="within")
    whisker_utility = w_util.groupby(["hour", "utility_id"], as_index=False).agg(whisker_events=("type", "size"), whisker_sensors=("sensors", "sum"))
else:
    whisker_utility = pd.DataFrame(columns=["hour", "utility_id", "whisker_events", "whisker_sensors"])

if "eagle_hourly" in globals() and isinstance(eagle_hourly, pd.DataFrame) and len(eagle_hourly) > 0:
    eh = eagle_hourly.copy()
    eh["hour"] = _as_hour_utc(eh["hour"])
    eh["county_fips"] = eh["county_fips"].astype(str).str.zfill(5)
    eh["eagle_customers_out"] = pd.to_numeric(eh["eagle_customers_out"], errors="coerce").fillna(0.0)
else:
    eh = pd.DataFrame(columns=["hour", "county_fips", "eagle_customers_out"])

eagle_utility = pd.DataFrame(columns=["hour", "utility_id", "eagle_customers_out"])
if len(eh) > 0 and county_gdf is not None and len(utility_gdf) > 0:
    county_eq = county_gdf.to_crs("EPSG:5070")
    util_eq = utility_gdf.to_crs("EPSG:5070")
    c_overlay = gpd.overlay(county_eq[["county_fips", "geometry"]], util_eq[["utility_id", "geometry"]], how="intersection")
    if len(c_overlay) > 0:
        c_overlay["int_area"] = c_overlay.geometry.area
        c_area = county_eq[["county_fips", "geometry"]].copy()
        c_area["county_area"] = c_area.geometry.area
        alloc = c_overlay.merge(c_area[["county_fips", "county_area"]], on="county_fips", how="left")
        alloc["w"] = alloc["int_area"] / alloc["county_area"]
        alloc = alloc[["county_fips", "utility_id", "w"]]
        eagle_utility = (
            eh.merge(alloc, on="county_fips", how="left")
            .assign(eagle_alloc=lambda d: d["eagle_customers_out"] * d["w"].fillna(0.0))
            .groupby(["hour", "utility_id"], as_index=False)
            .agg(eagle_customers_out=("eagle_alloc", "sum"))
        )


def _finalize_panel(haz_df, whisk_df, eagle_df, key):
    if haz_df is None:
        return None
    regions = sorted(pd.Series(haz_df[key].astype(str).unique()).dropna().tolist())
    base = pd.MultiIndex.from_product([event_hours, regions], names=["hour", key]).to_frame(index=False)

    keep_cols = ["hour", key, "hazard_index"] + (["utility_name"] if "utility_name" in haz_df.columns else [])
    out = base.merge(haz_df[keep_cols], on=["hour", key], how="left")
    if whisk_df is not None and len(whisk_df) > 0:
        out = out.merge(whisk_df, on=["hour", key], how="left")
    if eagle_df is not None and len(eagle_df) > 0:
        out = out.merge(eagle_df, on=["hour", key], how="left")

    for c in ["whisker_events", "whisker_sensors", "eagle_customers_out"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    out["hazard_index"] = out.groupby(key)["hazard_index"].transform(lambda s: s.interpolate(limit_direction="both"))

    out["target_whisker_spike"] = 0
    out["target_eagle_spike"] = 0
    if "whisker_events" in out.columns:
        thr_w = out.groupby(key)["whisker_events"].transform(lambda s: s.quantile(0.90))
        out["target_whisker_spike"] = (out["whisker_events"] >= thr_w).astype(int)
    if "eagle_customers_out" in out.columns:
        thr_e = out.groupby(key)["eagle_customers_out"].transform(lambda s: s.quantile(0.90))
        out["target_eagle_spike"] = (out["eagle_customers_out"] >= thr_e).astype(int)

    return out


chron_utility = _finalize_panel(haz_utility, whisker_utility, eagle_utility, "utility_id")
chron_county = _finalize_panel(haz_county, None, eh.rename(columns={"county_fips": "county_fips"}) if len(eh) > 0 else None, "county_fips") if haz_county is not None else None
chron_state = _finalize_panel(haz_state, None, None, "state_id") if haz_state is not None else None

if chron_utility is not None:
    util_name_ref = utility_gdf[["utility_id", "utility_name"]].drop_duplicates()
    chron_utility = chron_utility.merge(util_name_ref, on="utility_id", how="left", suffixes=("", "_ref"))
    if "utility_name_ref" in chron_utility.columns:
        chron_utility["utility_name"] = chron_utility["utility_name"].fillna(chron_utility["utility_name_ref"])
        chron_utility = chron_utility.drop(columns=["utility_name_ref"])

chronology_by_level = {
    "utility": chron_utility,
    "county": chron_county,
    "state": chron_state,
}

print("Chronology panels built.")
print({
    "utility_rows": 0 if chron_utility is None else int(len(chron_utility)),
    "utility_count": 0 if chron_utility is None else int(chron_utility["utility_id"].nunique()),
    "utility_name_col_present": bool(chron_utility is not None and "utility_name" in chron_utility.columns),
    "sample_utility_names": [] if chron_utility is None else chron_utility["utility_name"].dropna().astype(str).drop_duplicates().head(5).tolist(),
    "county_rows": 0 if chron_county is None else int(len(chron_county)),
    "county_count": 0 if chron_county is None else int(chron_county["county_fips"].nunique()),
    "hazard_vars": {"convective": conv_var, "wildfire": wf_var, "space_weather": sw_var},
})

print("Use chronology_by_level['utility'|'county'|'state'] for stratified/local model fitting.")

In [ ]:
# Plot fallback for Step 3f with AOI inset maps per utility
if "chron_utility" not in globals() or chron_utility is None or len(chron_utility) == 0:
    raise RuntimeError("chron_utility is empty. Re-run Cell 39 first.")
if "utility_gdf" not in globals() or utility_gdf is None or len(utility_gdf) == 0:
    raise RuntimeError("utility_gdf is missing. Re-run Cell 39 first.")

cu = chron_utility.copy()
cu["hour"] = pd.to_datetime(cu["hour"], errors="coerce", utc=True)
cu = cu.dropna(subset=["hour", "utility_id"]).copy()

ug = utility_gdf.copy()
if ug.crs is None:
    ug = ug.set_crs("EPSG:4326")
ug = ug.to_crs("EPSG:4326")

# Prefer human-readable utility label if available; otherwise use utility_id
name_col = None
for cand in ["utility_name", "name", "NAME", "UtilityName", "utilityname", "UTILITY_NAME", "Company", "company", "Provider", "provider", "UTILITY", "utility_id"]:
    if cand in ug.columns:
        name_col = cand
        break
if name_col is None:
    ug["utility_label"] = ug["utility_id"].astype(str)
else:
    ug["utility_label"] = ug[name_col].astype(str)

# Choose ranking metric with graceful fallback
rank_metric = None
for cand in ["eagle_customers_out", "whisker_events", "hazard_index"]:
    if cand in cu.columns:
        rank_metric = cand
        break
if rank_metric is None:
    raise RuntimeError("No ranking/plot columns found in chron_utility.")

ranked = (
    cu.groupby("utility_id", as_index=False)[rank_metric]
    .sum()
    .sort_values(rank_metric, ascending=False)
)
top_util = ranked.head(4)["utility_id"].tolist()
if len(top_util) == 0:
    raise RuntimeError("No utilities available to plot after filtering.")

# AOI for inset context (prefer analysis AOI if present)
if "gdf_aoi" in globals() and isinstance(gdf_aoi, gpd.GeoDataFrame) and len(gdf_aoi) > 0:
    aoi_plot = gdf_aoi.to_crs("EPSG:4326")
else:
    aoi_plot = None

fig, axes = plt.subplots(len(top_util), 1, figsize=(15, 3.2 * len(top_util)), sharex=True)
if len(top_util) == 1:
    axes = [axes]

for i, u in enumerate(top_util):
    d = cu[cu["utility_id"] == u].sort_values("hour")

    if "hazard_index" in d.columns:
        axes[i].plot(d["hour"], d["hazard_index"], color="tab:orange", linewidth=1.9, label="hazard_index")
    if "whisker_events" in d.columns:
        axes[i].plot(d["hour"], d["whisker_events"], color="tab:blue", alpha=0.72, linewidth=1.2, label="whisker_events")
    if "eagle_customers_out" in d.columns:
        axes[i].plot(d["hour"], d["eagle_customers_out"], color="black", alpha=0.72, linewidth=1.2, label="eagle_customers_out")

    urow = ug[ug["utility_id"].astype(str) == str(u)]
    if len(urow) > 0:
        ulabel = urow["utility_label"].iloc[0]
    else:
        ulabel = str(u)

    axes[i].set_title(f"Utility chronology: {ulabel} (ranked by {rank_metric})")
    axes[i].grid(alpha=0.2)
    axes[i].legend(loc="upper left", ncol=3, fontsize=8)

    # Inset map showing AOI + selected utility polygon
    ins = axes[i].inset_axes([0.78, 0.12, 0.2, 0.76])
    if aoi_plot is not None:
        aoi_plot.boundary.plot(ax=ins, color="0.4", linewidth=0.6)

    # Draw all utilities lightly for context, then selected utility in red
    try:
        ug.boundary.plot(ax=ins, color="0.75", linewidth=0.25, alpha=0.7)
    except Exception:
        pass

    if len(urow) > 0:
        urow.boundary.plot(ax=ins, color="crimson", linewidth=1.3)
        try:
            minx, miny, maxx, maxy = urow.total_bounds
            padx = (maxx - minx) * 0.5 + 0.2
            pady = (maxy - miny) * 0.5 + 0.2
            ins.set_xlim(minx - padx, maxx + padx)
            ins.set_ylim(miny - pady, maxy + pady)
        except Exception:
            pass

    ins.set_xticks([])
    ins.set_yticks([])
    ins.set_title("AOI inset", fontsize=7)

axes[-1].set_xlabel("UTC hour")
plt.tight_layout()
plt.show()

print({
    "plotted_utilities": len(top_util),
    "top_utilities": [str(x) for x in top_util],
    "ranking_metric": rank_metric,
    "label_column": name_col if name_col is not None else "utility_id",
    "has_aoi_inset": aoi_plot is not None,
})

## Step 3g: Spatio-Temporal Propagation and Collective Behavior

Track event emergence and spread in hazards, Whisker, and Eagle-I.

This cell estimates, per hour:
- active footprint size,
- number of spatial clusters,
- largest cluster size,
- cluster centroid motion (km/hour),

to diagnose whether collective behavior emerges and propagates across space.

In [ ]:
from scipy import ndimage
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

if "gridded_hazards" not in globals():
    raise RuntimeError("gridded_hazards is missing. Run hazard ingest cells first.")


def _as_hour_utc(x):
    return pd.DatetimeIndex(pd.to_datetime(x, errors="coerce", utc=True)).floor("h")


def _clean_cube(a):
    x = np.asarray(a, dtype=float)
    x[x <= -900] = np.nan
    return x


def _qbin(a, q=0.80):
    out = np.zeros(a.shape, dtype=np.uint8)
    good = np.isfinite(a)
    if good.any():
        thr = np.nanquantile(a[good], q)
        out[good] = (a[good] >= thr).astype(np.uint8)
    return out


def _haversine_km(lon1, lat1, lon2, lat2):
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = p2 - p1
    dl = np.radians(lon2 - lon1)
    aa = np.sin(dp / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2.0) ** 2
    return 2.0 * r * np.arcsin(np.sqrt(aa))


def _cluster_point_cloud(xy, radius_m=50000.0):
    n = 0 if xy is None else len(xy)
    if n == 0:
        return {"n_clusters": 0, "largest_cluster": 0, "labels": np.array([], dtype=int)}
    if n == 1:
        return {"n_clusters": 1, "largest_cluster": 1, "labels": np.array([0], dtype=int)}

    tree = cKDTree(xy)
    pairs = tree.query_pairs(r=radius_m)
    if len(pairs) == 0:
        labels = np.arange(n, dtype=int)
        return {"n_clusters": int(n), "largest_cluster": 1, "labels": labels}

    ii, jj = zip(*pairs)
    data = np.ones(len(ii), dtype=np.uint8)
    adj = coo_matrix((data, (ii, jj)), shape=(n, n))
    adj = adj + adj.T
    ncomp, labels = connected_components(adj, directed=False)
    counts = pd.Series(labels).value_counts()
    return {"n_clusters": int(ncomp), "largest_cluster": int(counts.max()), "labels": labels}


# ---- A) Hazard propagation from gridded multi-hazard activation ----
ds_tw = gridded_hazards.get("terrestrial_weather_gridded")
ds_wf = gridded_hazards.get("wildfire_gridded")
ds_sw = gridded_hazards.get("space_weather_gridded")

if ds_tw is None:
    raise RuntimeError("terrestrial_weather_gridded missing")

conv_var = "convective_proxy" if "convective_proxy" in ds_tw.data_vars else list(ds_tw.data_vars)[0]
wf_var = None
if ds_wf is not None and len(ds_wf.data_vars) > 0:
    wf_var = "frp" if "frp" in ds_wf.data_vars else list(ds_wf.data_vars)[0]
sw_var = None
if ds_sw is not None and len(ds_sw.data_vars) > 0:
    sw_var = "Eh" if "Eh" in ds_sw.data_vars else ("Bh" if "Bh" in ds_sw.data_vars else list(ds_sw.data_vars)[0])

conv = _qbin(_clean_cube(ds_tw[conv_var].values), q=0.80)
cubes = [conv]
if wf_var is not None:
    cubes.append(_qbin(_clean_cube(ds_wf[wf_var].values), q=0.80))
if sw_var is not None:
    cubes.append(_qbin(_clean_cube(ds_sw[sw_var].values), q=0.80))

ntime = min(c.shape[0] for c in cubes)
cubes = [c[:ntime] for c in cubes]
combo = (np.sum(np.stack(cubes, axis=0), axis=0) >= 2).astype(np.uint8)

haz_time = _as_hour_utc(ds_tw["time"].values[:ntime])
lat2d = ds_tw["lat"].values
lon2d = ds_tw["lon"].values

haz_rows = []
for t in range(ntime):
    m = combo[t]
    n_active = int(m.sum())
    if n_active == 0:
        haz_rows.append({"hour": haz_time[t], "source": "hazard", "n_active": 0, "n_clusters": 0, "largest_cluster": 0, "centroid_lat": np.nan, "centroid_lon": np.nan})
        continue

    lbl, ncl = ndimage.label(m, structure=np.ones((3, 3), dtype=int))
    if ncl > 0:
        sizes = ndimage.sum(m, lbl, index=np.arange(1, ncl + 1))
        largest = int(np.max(sizes))
    else:
        largest = 0

    c_lat = float(np.nanmean(lat2d[m > 0]))
    c_lon = float(np.nanmean(lon2d[m > 0]))

    haz_rows.append({
        "hour": haz_time[t],
        "source": "hazard",
        "n_active": n_active,
        "n_clusters": int(ncl),
        "largest_cluster": largest,
        "centroid_lat": c_lat,
        "centroid_lon": c_lon,
    })

haz_prop = pd.DataFrame(haz_rows).sort_values("hour")


# ---- B) Whisker propagation from point events ----
if "whisker_df" in globals() and isinstance(whisker_df, pd.DataFrame) and len(whisker_df) > 0:
    w = whisker_df.copy()
    w["hour"] = _as_hour_utc(w["start_utc"]) if "start_utc" in w.columns else _as_hour_utc(w["hour"])
    w = w.dropna(subset=["lat", "lon", "hour"])
else:
    w = pd.DataFrame(columns=["hour", "lat", "lon", "sensors"])

w_rows = []
for h, d in w.groupby("hour"):
    g = gpd.GeoDataFrame(d, geometry=gpd.points_from_xy(d["lon"], d["lat"]), crs="EPSG:4326").to_crs("EPSG:5070")
    xy = np.column_stack([g.geometry.x.values, g.geometry.y.values]) if len(g) > 0 else np.empty((0, 2))

    cl = _cluster_point_cloud(xy, radius_m=50000.0)
    if len(d) > 0:
        c_lat = float(np.average(d["lat"].values, weights=np.clip(pd.to_numeric(d.get("sensors", 1.0), errors="coerce").fillna(1.0).values, 1.0, None)))
        c_lon = float(np.average(d["lon"].values, weights=np.clip(pd.to_numeric(d.get("sensors", 1.0), errors="coerce").fillna(1.0).values, 1.0, None)))
    else:
        c_lat, c_lon = np.nan, np.nan

    w_rows.append({
        "hour": h,
        "source": "whisker",
        "n_active": int(len(d)),
        "n_clusters": cl["n_clusters"],
        "largest_cluster": cl["largest_cluster"],
        "centroid_lat": c_lat,
        "centroid_lon": c_lon,
    })

whisker_prop = pd.DataFrame(w_rows)


# ---- C) Eagle-I propagation from active counties ----
if "eagle_hourly" in globals() and isinstance(eagle_hourly, pd.DataFrame) and len(eagle_hourly) > 0:
    e = eagle_hourly.copy()
    e["hour"] = _as_hour_utc(e["hour"])
    e["county_fips"] = e["county_fips"].astype(str).str.zfill(5)
    e["eagle_customers_out"] = pd.to_numeric(e["eagle_customers_out"], errors="coerce").fillna(0.0)
else:
    e = pd.DataFrame(columns=["hour", "county_fips", "eagle_customers_out"])

# Prefer the county layer loaded in Cell 39, then fall back to earlier names if present
if "counties_gdf" in globals() and isinstance(counties_gdf, gpd.GeoDataFrame) and len(counties_gdf) > 0:
    c_ref = counties_gdf.to_crs("EPSG:4326").copy()
elif "county_gdf" in globals() and isinstance(county_gdf, gpd.GeoDataFrame) and len(county_gdf) > 0:
    c_ref = county_gdf.to_crs("EPSG:4326").copy()
elif "counties_aoi" in globals() and isinstance(counties_aoi, gpd.GeoDataFrame) and len(counties_aoi) > 0:
    c_ref = counties_aoi.to_crs("EPSG:4326").copy()
elif "counties" in globals() and isinstance(counties, gpd.GeoDataFrame) and len(counties) > 0:
    c_ref = counties.to_crs("EPSG:4326").copy()
else:
    c_ref = None

if c_ref is not None:
    c_fips_col = "county_fips" if "county_fips" in c_ref.columns else ("GEOID" if "GEOID" in c_ref.columns else ("COUNTYFP" if "COUNTYFP" in c_ref.columns else None))
    if c_fips_col is not None:
        c_ref["county_fips"] = c_ref[c_fips_col].astype(str).str.zfill(5)
        cent = c_ref.copy()
        cent["geometry"] = cent.geometry.centroid
        cent = cent[["county_fips", "geometry"]]
    else:
        cent = None
else:
    cent = None

print({
    "eagle_rows_in": int(len(e)),
    "county_layer_present": bool(c_ref is not None),
    "county_centroids_present": bool(cent is not None),
    "positive_eagle_rows": int((e["eagle_customers_out"] > 0).sum()) if len(e) > 0 else 0,
})

e_rows = []
if cent is not None and len(e) > 0:
    for h, d in e.groupby("hour"):
        d = d[d["eagle_customers_out"] > 0].copy()
        if len(d) == 0:
            e_rows.append({"hour": h, "source": "eagle", "n_active": 0, "n_clusters": 0, "largest_cluster": 0, "centroid_lat": np.nan, "centroid_lon": np.nan})
            continue

        dd = d.merge(cent, on="county_fips", how="left").dropna(subset=["geometry"])
        if len(dd) == 0:
            e_rows.append({"hour": h, "source": "eagle", "n_active": 0, "n_clusters": 0, "largest_cluster": 0, "centroid_lat": np.nan, "centroid_lon": np.nan})
            continue

        gg = gpd.GeoDataFrame(dd, geometry="geometry", crs="EPSG:4326")
        gg_m = gg.to_crs("EPSG:5070")
        xy = np.column_stack([gg_m.geometry.x.values, gg_m.geometry.y.values])

        # Use a wider radius than the point-level Whisker layer because counties are coarse units.
        cl = _cluster_point_cloud(xy, radius_m=180000.0)

        lat = gg.geometry.y.values
        lon = gg.geometry.x.values
        wgt = np.clip(dd["eagle_customers_out"].values.astype(float), 1.0, None)

        e_rows.append({
            "hour": h,
            "source": "eagle",
            "n_active": int(len(dd)),
            "n_clusters": cl["n_clusters"],
            "largest_cluster": cl["largest_cluster"],
            "centroid_lat": float(np.average(lat, weights=wgt)),
            "centroid_lon": float(np.average(lon, weights=wgt)),
        })

eagle_prop = pd.DataFrame(e_rows)


# ---- D) Merge and compute centroid-speed diagnostics ----
propagation_metrics = pd.concat([haz_prop, whisker_prop, eagle_prop], ignore_index=True)
propagation_metrics = propagation_metrics.sort_values(["source", "hour"]).copy()
propagation_metrics["centroid_shift_km"] = np.nan

for src in propagation_metrics["source"].dropna().unique():
    m = propagation_metrics["source"] == src
    d = propagation_metrics.loc[m].sort_values("hour").copy()
    if len(d) < 2:
        continue

    shift = [np.nan]
    for i in range(1, len(d)):
        a = d.iloc[i - 1]
        b = d.iloc[i]
        if np.isfinite(a["centroid_lat"]) and np.isfinite(a["centroid_lon"]) and np.isfinite(b["centroid_lat"]) and np.isfinite(b["centroid_lon"]):
            shift.append(float(_haversine_km(a["centroid_lon"], a["centroid_lat"], b["centroid_lon"], b["centroid_lat"])))
        else:
            shift.append(np.nan)
    propagation_metrics.loc[d.index, "centroid_shift_km"] = shift


# ---- E) Visual diagnostics ----
fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True, constrained_layout=True)

for src, clr in [("hazard", "tab:orange"), ("whisker", "tab:blue"), ("eagle", "black")]:
    d = propagation_metrics[propagation_metrics["source"] == src].sort_values("hour")
    if len(d) == 0:
        continue
    axes[0].plot(d["hour"], d["n_active"], label=src, color=clr)
    axes[1].plot(d["hour"], d["n_clusters"], label=src, color=clr)
    axes[2].plot(d["hour"], d["largest_cluster"], label=src, color=clr)
    axes[3].plot(d["hour"], d["centroid_shift_km"], label=src, color=clr)

axes[0].set_title("Active footprint size")
axes[1].set_title("Number of spatial clusters")
axes[2].set_title("Largest cluster size")
axes[3].set_title("Centroid movement (km/hour)")

for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(loc="upper left")

axes[-1].set_xlabel("UTC hour")
plt.show()

# Centroid trajectories
fig, ax = plt.subplots(1, 1, figsize=(9, 7))
for src, clr in [("hazard", "tab:orange"), ("whisker", "tab:blue"), ("eagle", "black")]:
    d = propagation_metrics[propagation_metrics["source"] == src].dropna(subset=["centroid_lat", "centroid_lon"]).sort_values("hour")
    if len(d) == 0:
        continue
    ax.plot(d["centroid_lon"], d["centroid_lat"], marker="o", markersize=2.5, linewidth=1.0, alpha=0.8, label=src, color=clr)

if "gdf_aoi" in globals() and isinstance(gdf_aoi, gpd.GeoDataFrame):
    gdf_aoi.to_crs("EPSG:4326").boundary.plot(ax=ax, color="gray", linewidth=0.6, alpha=0.5)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Propagation trajectories of activity centroids")
ax.grid(alpha=0.2)
ax.legend(loc="best")
plt.tight_layout()
plt.show()


def _summary_block(df, src):
    d = df[df["source"] == src].copy()
    if len(d) == 0:
        return {"source": src, "rows": 0}
    return {
        "source": src,
        "rows": int(len(d)),
        "peak_active": float(pd.to_numeric(d["n_active"], errors="coerce").max()),
        "peak_clusters": float(pd.to_numeric(d["n_clusters"], errors="coerce").max()),
        "peak_largest_cluster": float(pd.to_numeric(d["largest_cluster"], errors="coerce").max()),
        "median_centroid_shift_km": float(pd.to_numeric(d["centroid_shift_km"], errors="coerce").median(skipna=True)),
    }

print("Propagation summary:")
print(_summary_block(propagation_metrics, "hazard"))
print(_summary_block(propagation_metrics, "whisker"))
print(_summary_block(propagation_metrics, "eagle"))

print("Saved output dataframe: propagation_metrics")
print("Interpretation: rising n_clusters and rising largest_cluster followed by higher centroid_shift_km indicates emergence and spread of collective behavior.")

In [ ]:
# Compact results readout for hazard-importance study
if "hazard_obs_skill_best" not in globals():
    raise RuntimeError("hazard_obs_skill_best missing. Run the lagged scoring cell first.")

skill = hazard_obs_skill_best.copy()

print("Top predictors for Eagle-I spike target (best directional AUC):")
print(
    skill[skill["target"] == "target_eagle_spike"]
    [["predictor", "lag_h", "best_auc", "spearman_rho", "n"]]
    .head(10)
    .to_string(index=False)
)

print("\nTop predictors for Whisker spike target (best directional AUC):")
print(
    skill[skill["target"] == "target_whisker_spike"]
    [["predictor", "lag_h", "best_auc", "spearman_rho", "n"]]
    .head(10)
    .to_string(index=False)
)

# Mechanism-level summary (group channels into terrestrial mechanisms)
if "hazard_obs_skill_long" in globals():
    long_df = hazard_obs_skill_long.copy()

    def mech(p):
        p = str(p).lower()
        if "convective_proxy" in p or "reflectivity" in p or "shear" in p:
            return "convective_severity"
        if "wspd" in p or p.startswith("u10") or p.startswith("v10"):
            return "wind_loading"
        if "qpe" in p or "precip" in p or "flood" in p:
            return "precip_flooding"
        if "frp" in p or "wildfire" in p:
            return "wildfire"
        if "eh" in p or "bh" in p or "space" in p:
            return "space_weather"
        return "other"

    long_df["mechanism"] = long_df["predictor"].map(mech)
    mech_summary = (
        long_df.groupby(["target", "mechanism"], as_index=False)
        .agg(mean_best_auc=("best_auc", "mean"), max_best_auc=("best_auc", "max"), mean_abs_rho=("spearman_rho", lambda s: s.abs().mean()))
        .sort_values(["target", "max_best_auc"], ascending=[True, False])
    )
    print("\nMechanism-level summary (aggregated across channels):")
    print(mech_summary.to_string(index=False))

# Single vs multi-hazard comparison
if "interaction_df" in globals() and interaction_df is not None and len(interaction_df) > 0:
    print("\nMulti-hazard interaction diagnostics:")
    show_cols = [c for c in ["interaction", "rate", "baseline", "lift"] if c in interaction_df.columns]
    print(interaction_df[show_cols].sort_values("interaction").to_string(index=False))

## Step 3h: Hourly Cluster Movie

Render an hourly movie of the cluster evolution for hazard, Whisker, and Eagle-I.

The animation keeps the same AOI frame and updates cluster membership every hour so propagation is easy to inspect as a movie.

In [ ]:
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation, FFMpegWriter

# Increase notebook animation embed limit (MB) to avoid dropped frames in long hourly movies
plt.rcParams["animation.embed_limit"] = 100.0

if "gridded_hazards" not in globals():
    raise RuntimeError("gridded_hazards is missing. Run the hazard ingest cells first.")
if "whisker_df" not in globals() or "eagle_hourly" not in globals():
    raise RuntimeError("whisker_df and eagle_hourly are required. Run the impact ingest cells first.")
if "counties_gdf" not in globals() or counties_gdf is None or len(counties_gdf) == 0:
    raise RuntimeError("counties_gdf is missing. Run Cell 39 first.")

# ---- Rebuild hourly cluster states ----
ds_tw = gridded_hazards.get("terrestrial_weather_gridded")
ds_wf = gridded_hazards.get("wildfire_gridded")
ds_sw = gridded_hazards.get("space_weather_gridded")

if ds_tw is None:
    raise RuntimeError("terrestrial_weather_gridded missing.")

conv_var = "convective_proxy" if "convective_proxy" in ds_tw.data_vars else list(ds_tw.data_vars)[0]
wf_var = "frp" if ds_wf is not None and "frp" in ds_wf.data_vars else (list(ds_wf.data_vars)[0] if ds_wf is not None and len(ds_wf.data_vars) > 0 else None)
sw_var = "Eh" if ds_sw is not None and "Eh" in ds_sw.data_vars else ("Bh" if ds_sw is not None and "Bh" in ds_sw.data_vars else (list(ds_sw.data_vars)[0] if ds_sw is not None and len(ds_sw.data_vars) > 0 else None))


def _clean_cube(a):
    arr = np.asarray(a, dtype=float)
    arr[arr <= -900] = np.nan
    return arr


def _qbin(a, q=0.80):
    out = np.zeros(a.shape, dtype=np.uint8)
    good = np.isfinite(a)
    if good.any():
        thr = np.nanquantile(a[good], q)
        out[good] = (a[good] >= thr).astype(np.uint8)
    return out


def _cluster_point_cloud(xy, radius_m):
    if xy is None or len(xy) == 0:
        return np.array([], dtype=int), 0
    if len(xy) == 1:
        return np.array([1], dtype=int), 1
    tree = cKDTree(xy)
    pairs = tree.query_pairs(r=radius_m)
    if len(pairs) == 0:
        return np.arange(1, len(xy) + 1, dtype=int), len(xy)
    ii, jj = zip(*pairs)
    data = np.ones(len(ii), dtype=np.uint8)
    adj = coo_matrix((data, (ii, jj)), shape=(len(xy), len(xy)))
    adj = adj + adj.T
    ncomp, labels = connected_components(adj, directed=False)
    return labels + 1, int(ncomp)


# Hour grid for the movie
event_hours = pd.date_range(EVENT_START.floor("h"), EVENT_END.floor("h"), freq="1h", tz="UTC")

# Shared AOI bounds
if "gdf_aoi" in globals() and isinstance(gdf_aoi, gpd.GeoDataFrame) and len(gdf_aoi) > 0:
    aoi_plot = gdf_aoi.to_crs("EPSG:4326")
    xmin, ymin, xmax, ymax = aoi_plot.total_bounds
else:
    aoi_plot = None
    xmin, ymin, xmax, ymax = -107.0, 24.0, -72.0, 50.5

# Precompute hazard cluster images
conv = _qbin(_clean_cube(ds_tw[conv_var].values), q=0.80)
cubes = [conv]
if wf_var is not None:
    cubes.append(_qbin(_clean_cube(ds_wf[wf_var].values), q=0.80))
if sw_var is not None:
    cubes.append(_qbin(_clean_cube(ds_sw[sw_var].values), q=0.80))
ntime = min(c.shape[0] for c in cubes)
conv = conv[:ntime]
cubes = [c[:ntime] for c in cubes]
combo = (np.sum(np.stack(cubes, axis=0), axis=0) >= 2).astype(np.uint8)
haz_times = pd.DatetimeIndex(pd.to_datetime(ds_tw["time"].values[:ntime], utc=True)).floor("h")
lat2d = ds_tw["lat"].values
lon2d = ds_tw["lon"].values
hazard_frames = []
for t in range(ntime):
    labels, ncl = ndimage.label(combo[t], structure=np.ones((3, 3), dtype=int))
    hazard_frames.append({
        "hour": haz_times[t],
        "labels": labels.astype(int),
        "n_clusters": int(ncl),
    })

# Precompute Whisker cluster points
whisker_local = whisker_df.copy()
whisker_local["hour"] = pd.to_datetime(whisker_local["start_utc"], errors="coerce", utc=True).dt.floor("h")
whisker_local = whisker_local.dropna(subset=["hour", "lat", "lon"]).copy()
whisker_frames = []
for hour, d in whisker_local.groupby("hour"):
    gg = gpd.GeoDataFrame(d, geometry=gpd.points_from_xy(d["lon"], d["lat"]), crs="EPSG:4326").to_crs("EPSG:5070")
    xy = np.column_stack([gg.geometry.x.values, gg.geometry.y.values]) if len(gg) > 0 else np.empty((0, 2))
    labels, ncl = _cluster_point_cloud(xy, radius_m=50000.0)
    gg_plot = gpd.GeoDataFrame(d.copy(), geometry=gpd.points_from_xy(d["lon"], d["lat"]), crs="EPSG:4326")
    gg_plot["cluster_id"] = labels if len(labels) == len(gg_plot) else np.array([], dtype=int)
    whisker_frames.append({"hour": hour, "gdf": gg_plot, "n_clusters": int(ncl)})
whisker_frame_map = {item["hour"]: item for item in whisker_frames}

# Precompute Eagle-I county centroids and cluster points
if "county_gdf" in globals() and isinstance(county_gdf, gpd.GeoDataFrame) and len(county_gdf) > 0:
    county_eq = county_gdf.to_crs("EPSG:5070").copy()
    county_eq["geometry"] = county_eq.geometry.centroid
    county_pts = county_eq.to_crs("EPSG:4326")[["county_fips", "geometry"]].copy()
else:
    county_pts = None

eagle_local = eagle_hourly.copy()
eagle_local["hour"] = pd.to_datetime(eagle_local["hour"], errors="coerce", utc=True).dt.floor("h")
eagle_local["county_fips"] = eagle_local["county_fips"].astype(str).str.zfill(5)
eagle_local["eagle_customers_out"] = pd.to_numeric(eagle_local["eagle_customers_out"], errors="coerce").fillna(0.0)
eagle_local = eagle_local[eagle_local["eagle_customers_out"] > 0].copy()
eagle_frames = []
for hour, d in eagle_local.groupby("hour"):
    if county_pts is None:
        continue
    dd = d.merge(county_pts, on="county_fips", how="left").dropna(subset=["geometry"])
    if len(dd) == 0:
        continue
    gg = gpd.GeoDataFrame(dd, geometry="geometry", crs="EPSG:4326").to_crs("EPSG:5070")
    xy = np.column_stack([gg.geometry.x.values, gg.geometry.y.values])
    labels, ncl = _cluster_point_cloud(xy, radius_m=180000.0)
    gg_plot = gpd.GeoDataFrame(dd.copy(), geometry="geometry", crs="EPSG:4326")
    gg_plot["cluster_id"] = labels if len(labels) == len(gg_plot) else np.array([], dtype=int)
    eagle_frames.append({"hour": hour, "gdf": gg_plot, "n_clusters": int(ncl)})
eagle_frame_map = {item["hour"]: item for item in eagle_frames}

print({
    "movie_hours": len(event_hours),
    "hazard_frames": len(hazard_frames),
    "whisker_hours": len(whisker_frame_map),
    "eagle_hours": len(eagle_frame_map),
})

# Build movie figure
cmap = plt.get_cmap("tab20", 20)
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
for ax in axes:
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.15)
    if aoi_plot is not None:
        aoi_plot.boundary.plot(ax=ax, color="black", linewidth=0.6, alpha=0.7)

haz_im = axes[0].imshow(
    np.zeros_like(hazard_frames[0]["labels"], dtype=float),
    origin="upper",
    extent=[lon2d.min(), lon2d.max(), lat2d.min(), lat2d.max()],
    cmap="tab20",
    interpolation="nearest",
    alpha=0.95,
)
axes[0].set_title("Hazard clusters")

whisker_sc = axes[1].scatter([], [], s=[], c=[], cmap=cmap, vmin=0, vmax=20, edgecolor="none", alpha=0.85)
axes[1].set_title("Whisker clusters")

eagle_sc = axes[2].scatter([], [], s=[], c=[], cmap=cmap, vmin=0, vmax=20, edgecolor="none", alpha=0.85)
axes[2].set_title("Eagle-I county clusters")


def _frame_colors(labels):
    if len(labels) == 0:
        return np.array([])
    return (labels.astype(int) % 20).astype(float)


def _frame_sizes(n):
    if n == 0:
        return np.array([])
    return np.full(n, 35.0)


def update(frame_idx):
    hour = event_hours[min(frame_idx, len(event_hours) - 1)]
    title_ts = pd.Timestamp(hour).strftime("%Y-%m-%d %H:%M UTC")

    hf = hazard_frames[min(frame_idx, len(hazard_frames) - 1)]
    labels = hf["labels"]
    hazard_map = np.where(labels > 0, labels, np.nan)
    haz_im.set_data(hazard_map)
    finite = hazard_map[np.isfinite(hazard_map)]
    vmax = int(np.max(finite)) if finite.size > 0 else 2
    haz_im.set_clim(vmin=1, vmax=max(2, vmax))
    axes[0].set_title(f"Hazard clusters | {title_ts} | n={hf['n_clusters']}")

    wf = whisker_frame_map.get(hour)
    if wf is not None and len(wf["gdf"]) > 0:
        gg = wf["gdf"]
        whisker_sc.set_offsets(np.column_stack([gg.geometry.x.values, gg.geometry.y.values]))
        whisker_sc.set_array(_frame_colors(gg["cluster_id"].values))
        whisker_sc.set_sizes(_frame_sizes(len(gg)))
    else:
        whisker_sc.set_offsets(np.empty((0, 2)))
        whisker_sc.set_array(np.array([]))
        whisker_sc.set_sizes(np.array([]))
    axes[1].set_title(f"Whisker clusters | {title_ts} | n={0 if wf is None else wf['n_clusters']}")

    ef = eagle_frame_map.get(hour)
    if ef is not None and len(ef["gdf"]) > 0:
        gg = ef["gdf"]
        eagle_sc.set_offsets(np.column_stack([gg.geometry.x.values, gg.geometry.y.values]))
        eagle_sc.set_array(_frame_colors(gg["cluster_id"].values))
        eagle_sc.set_sizes(_frame_sizes(len(gg)))
    else:
        eagle_sc.set_offsets(np.empty((0, 2)))
        eagle_sc.set_array(np.array([]))
        eagle_sc.set_sizes(np.array([]))
    axes[2].set_title(f"Eagle-I county clusters | {title_ts} | n={0 if ef is None else ef['n_clusters']}")

    return haz_im, whisker_sc, eagle_sc

anim = FuncAnimation(fig, update, frames=len(event_hours), interval=220, blit=False, repeat=True)
plt.close(fig)

display(HTML(anim.to_jshtml()))

# Optional MP4 export (requires ffmpeg in environment). Set to False to skip writing.
SAVE_MP4 = False
movie_path = Path("data/cluster_movie_hourly.mp4")
if SAVE_MP4:
    movie_path.parent.mkdir(parents=True, exist_ok=True)
    writer = FFMpegWriter(fps=6, bitrate=1800)
    anim.save(str(movie_path), writer=writer)
    print(f"Saved movie to: {movie_path}")

print({
    "frames": len(event_hours),
    "hourly_resolution": True,
    "sources": ["hazard", "whisker", "eagle"],
    "mp4_save_enabled": SAVE_MP4,
})